# TinyLlama Fine-Tuned for Python
### Teaching a small language model to answer Python questions with clean code, on a free Colab T4 GPU

---

## What this notebook does, from start to finish

We take a ready-made language model called **TinyLlama** and train it a little further so that it changes **how** it answers coding questions.

**Before** our training, if you ask TinyLlama *"Write a Python function that reverses a string"*, it replies like a chatty assistant:

- "Here's a Python function that reverses a string:"
- then some code
- then "Here's an example usage:"
- then more examples and notes
- and it often gets cut off mid-sentence because it never decides to stop.

**After** our training, it replies with just the code, and then it stops.

That change in behaviour is what **fine-tuning** means in this notebook: we keep everything the model already knows, and adjust a small part of it so that it follows a new habit.

---

## The journey, part by part

Each part below says **what goes in**, **what we do**, and **what comes out**.

| Part | What goes in | What we do | What comes out |
|---|---|---|---|
| **0. Key ideas** | Nothing to run | Read three short notes: what a model is in numbers, what quantization is, and common questions about it | The ideas you need for the rest of the notebook |
| **1. What fine-tuning means** | Nothing to run | Read what fine-tuning is, with a before/after example | A clear goal: change the model's answering habit |
| **2. Setup** | An empty Colab session | 2.1 Check the GPU is on. 2.2 Install 5 libraries. 2.3 Set every setting (number of examples, LoRA size, learning rate) in one cell | A GPU-ready session and one cell of settings you can change with sliders |
| **3. Meet TinyLlama** | The model name | 3.1 Load the **tokenizer** and see how text turns into numbers. 3.2 Fix the **padding token** so the model can learn when to stop. 3.3 Load the model in **4-bit** form so it fits in memory | A model on the GPU using about 0.7 GB instead of about 4 GB |
| **4. Before** | 5 fixed Python questions | 4.1 See the exact chat layout the model expects. 4.2 Write an `ask` helper function. 4.3 Ask the untouched model all 5 questions | 5 "before" answers saved for comparison |
| **5. Training data** | A public dataset of 18,612 (instruction, Python code) pairs | 5.1 Download and look at rows. 5.2 Rewrite each row into the chat layout with the answer filled in. 5.3 Convert to tokens and drop very long rows. 5.4 Check every example ends with the stop token | 2,000 training examples and 100 separate checking examples |
| **6. LoRA** | The 4-bit model | Freeze all 1.1 billion original numbers. Attach small trainable tables (LoRA) next to them | A model where only about 1% of the numbers can change |
| **7. Training** | The model + 2,000 examples | 7.1 Understand one training step and the **loss**. 7.2 Set the training settings. 7.3 Measure the starting loss. 7.4 Train. 7.5 Chart the loss | A trained model and a loss that has dropped |
| **8. After** | The same 5 questions | Ask the trained model, show before and after together, and compare answer lengths | Visible proof of the new habit |
| **9. Save** | The trained LoRA tables | 9.1 Save them as an **adapter**. 9.2 Load them back onto a fresh copy of TinyLlama. 9.3 Optionally download or upload them | An adapter folder of about 50 MB |
| **10. Your turn** | Any Python question you type | Ask the trained model live | Its answer |
| **11. Recap** | Nothing to run | Summary, limits, glossary, exercises | |

---

## What we got in our own test run on a Colab T4

Your numbers will be close to these, not identical.

| Measurement | Result |
|---|---|
| GPU memory used by the 4-bit model | 0.70 GB |
| Numbers trained by LoRA | 12,615,680 (1.13% of the model) |
| Training time for 2,000 examples, 1 pass | 6.6 minutes |
| Loss on the checking rows before training | 1.514 |
| Loss on the checking rows after training | 0.585 |
| Length of the 5 answers before → after | between 420 and 693 characters → between 121 and 294 characters |
| Size of the saved adapter | about 48 MB (the full model is 2.2 GB) |

---

## The techniques you will see, in one line each

| Technique | One-line meaning | Where |
|---|---|---|
| Tokenization | Cutting text into pieces and giving each piece an ID number | Part 3.1 |
| Padding fix | Making sure the "I am finished" token is not ignored during training | Part 3.2 |
| Quantization (4-bit) | Storing each of the model's numbers in 4 bits instead of 32 | Part 3.3 |
| Chat template | Writing every example in the exact layout the model was trained on | Parts 4 and 5 |
| Train / checking split | Keeping 100 examples aside to test on rows the model never learned from | Part 5.3 |
| LoRA | Freezing the model and training small extra tables next to it | Part 6 |
| QLoRA | Quantization and LoRA used together | Parts 3 and 6 |
| Loss | One number that measures how wrong the predictions are | Part 7 |
| Adapter | The saved LoRA tables, kept separate from the model | Part 9 |

---

## How to use this notebook

1. In the Colab menu choose **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the cells **from top to bottom, one at a time** (Shift + Enter).
3. Every code cell starts with a block explaining what the cell does, and every line of code has its own explanation above it.
4. Look for these boxes as you go:
   - **Predict first:** guess the result before running the cell.
   - **Try it:** change something small and run the cell again.
   - **Check yourself:** a question with a hidden answer you can click to open.

---
# Part 0 — Key ideas before we start

## What a model is, in numbers

TinyLlama is about 1.1 billion numbers stored in a file. These numbers are called **parameters** or **weights**.

When you type a question, the text is turned into numbers.

Those numbers are multiplied with the parameters, layer after layer. The result is the next word, and this repeats word by word.

**Training** means changing the parameters so that the result comes out closer to the answer we want.

## Quantization

The problem is memory.

Each parameter is normally stored using 32 bits (4 bytes).
1.1 billion × 4 bytes = **4.4 GB**, just to hold the model.

Training needs several times more memory on top of that. The free T4 GPU has about 15 GB.

**Quantization stores each parameter using fewer bits. We use 4 bits.**

With 32 bits, a parameter can be almost any value, for example 0.037218.

With 4 bits, only **16 different values** exist. So 0.037218 is replaced by the nearest of those 16 values, for example 0.04.
(A byte is 8 bits, so 4 bits is half a byte. The 16 comes straight from that.)

### Where "16" comes from

A bit can only be 0 or 1. With 4 bits, these are all the patterns you can write:

```
0000  0001  0010  0011  0100  0101  0110  0111
1000  1001  1010  1011  1100  1101  1110  1111
```

That is 2 × 2 × 2 × 2 = 16 patterns, and nothing else is possible. So a 4-bit slot can hold one of 16 codes, numbered 0 to 15.

Each parameter becomes slightly less exact, and takes 8 times less space.

The model drops from about 4.4 GB to well under 1 GB.

When the GPU calculates, it temporarily expands those numbers to 16 bits, does the multiplication, and keeps storing them in 4 bits. That is what `bnb_4bit_compute_dtype=torch.float16` means in the notebook.

### Quantization is about memory: smaller storage, slightly less precision.

## Common questions about quantization

**1. When should we use quantization?**
Use it when the model does not fit in your GPU memory, or when you want it to run faster and cheaper. Common cases are fine-tuning on a small GPU (like our T4) and running a model on a laptop, phone or low-cost server.

**2. When should we not use it?**
Skip it when you have enough GPU memory and need the highest accuracy, for example in tasks where small errors matter, like exact maths or finance calculations. Also be careful with already-small models: a 1B model loses more quality from 4-bit than a 70B model does.

**3. Does quantization make the model worse?**
Slightly. Each weight is rounded to the nearest of 16 levels, so there is a small loss of accuracy. 8-bit usually loses almost nothing, and 4-bit (NF4) loses a little. Always compare answers before and after on your own test questions.

**4. If the model is in 4-bit, how can we train it?**
We don't train the 4-bit weights. They stay frozen. We train only the small LoRA tables, which are kept in normal precision. That combination is QLoRA.

**5. Is quantization done during training or after?**
Both are possible. In our notebook it happens at load time: the model is converted to 4-bit as it loads, and then we fine-tune on top (QLoRA). Quantization can also be applied after training to make the finished model smaller to deploy.

---
# Part 1 — What does "fine-tuning" mean?

## What we are doing in this part

Nothing runs here. We set a clear goal for the whole notebook, so every later step has a reason.

## Pre-training and fine-tuning

A language model is a very large list of numbers. These numbers are called **parameters** (or **weights**). TinyLlama has about 1,100,000,000 of them.

The model does one job: given some text, it predicts the next small piece of text. The parameters decide which piece it predicts.

- **Pre-training** is how those numbers were first set. The TinyLlama team showed the model a huge amount of general text, about 3 trillion tokens, and adjusted the numbers until its predictions were good. Their plan used 16 large GPUs for about 90 days.
- **Fine-tuning** is what we do today. We take the finished model and keep adjusting its numbers a little more, using a small set of examples of one specific task. Our task is: *an instruction goes in, Python code comes out*.

| | Pre-training | Our fine-tuning |
|---|---|---|
| Starting point | Random numbers | The finished TinyLlama |
| Training text | About 3 trillion tokens of general text | 2,000 (instruction, Python code) pairs |
| Hardware | 16 large GPUs | 1 free Colab T4 |
| Time | Months | About 7 minutes in our test run |
| Goal | Learn language in general | Learn one habit: answer with code, then stop |

## A concrete example of what we want to change

Question: `Write a Python function that reverses a string.`

| | A typical answer |
|---|---|
| A general chat model | "Here's a Python function that reverses a string:" followed by code, followed by "Here's an example usage:", followed by more notes, often cut off mid-sentence |
| What we want | `def reverse_string(s):`<br>`    return s[::-1]` |

We are not adding new knowledge about Python from nowhere. TinyLlama has already seen code during pre-training. We are changing its **habit**: answer with code, and stop when the code is finished.

## What fine-tuning needs

1. **A base model** — TinyLlama (Part 3).
2. **Examples** — pairs of (instruction, correct Python code). We use 2,000 of them (Part 5).
3. **A training method** — we use **QLoRA**: a 4-bit model (Part 3.3) plus small trainable tables (Part 6). It lets us train on a free GPU.
4. **A way to check the result** — the same 5 questions before and after (Parts 4 and 8), and the loss number (Part 7).

<details>
<summary><b>Check yourself:</b> after fine-tuning, does TinyLlama know more Python than before? (click to open)</summary>

Mostly no. 2,000 examples seen once are far too few to teach a lot of new Python. What changes is the **format** of its answers: code only, and a clean stop. Correctness improves only a little.
</details>

---
# Part 2 — Setting up

## What we are doing in this part

We get the Colab session ready in three steps:

1. **2.1** Confirm a GPU is switched on.
2. **2.2** Install the five libraries we need and import the tools from them.
3. **2.3** Put every setting in one cell, with sliders you can change.

At the end of Part 2 nothing has been downloaded or trained yet. The session is simply ready.

## 2.1 Is the GPU switched on?

Training means doing billions of multiplications. A **GPU** does them many times faster than a normal processor (CPU). Colab's free GPU is the **T4**, which has about 15 GB of its own memory.

Run the cell below. If it says no GPU was found, go to **Runtime → Change runtime type → T4 GPU** and start again.

**Predict first:** the T4 is sold as a 16 GB card. Will the cell print 16 GB? (Run it and see. It prints about 14.6–14.7 GB, because the cell converts bytes using 1024 instead of 1000, and a small part of the memory is reserved.)

In [ ]:
# =============================================================================
# CELL: CHECK THAT COLAB HAS GIVEN US A GPU
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Training a model means doing billions of multiplications.
# A GPU (graphics card) does these multiplications far faster than the
# normal processor (CPU). Without a GPU, the later cells would be too slow
# or would fail.
#
# So this cell asks one question: "Is there a GPU?"
#   - If YES -> it prints the GPU's name and how much memory it has.
#               On Colab this should be "Tesla T4" with about 15 GB.
#   - If NO  -> it prints a message telling you how to switch the GPU on.
#
# WHY THE MEMORY NUMBER MATTERS
# -----------------------------
# The whole reason we use quantization and LoRA later in this notebook
# is to fit the training inside this roughly 15 GB of GPU memory.
#
# NOTE ON INDENTATION
# -------------------
# Python uses the spaces at the start of a line to know which lines belong
# inside the "if" and which belong inside the "else".
# If those spaces are missing, Python stops with an "IndentationError".
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: import torch
# -----------------------------------------------------------------------------
# - This loads a library called PyTorch into the notebook.
# - In code, PyTorch is called "torch".
# - PyTorch is the library that stores the model's numbers and does all the
#   maths on them. Every later step in this notebook depends on it.
# - Colab already has PyTorch installed, so nothing is downloaded here.
import torch


# -----------------------------------------------------------------------------
# LINE 2: if torch.cuda.is_available():
# -----------------------------------------------------------------------------
# - "CUDA" is NVIDIA's system that lets programs use an NVIDIA GPU.
#   The T4 is an NVIDIA GPU.
# - torch.cuda.is_available() gives back:
#       True  -> a GPU is connected and can be used
#       False -> no GPU is available
# - "if ... :" means: run the indented lines below ONLY when this is True.
if torch.cuda.is_available():

    # -------------------------------------------------------------------------
    # LINE 3: gpu_name = torch.cuda.get_device_name(0)
    # -------------------------------------------------------------------------
    # - Asks for the name of the GPU and stores it in a variable "gpu_name".
    # - The (0) means "the first GPU". Computers start counting at 0.
    #   Colab gives us one GPU, so it is GPU number 0.
    # - On Colab the result is usually the text "Tesla T4".
    gpu_name = torch.cuda.get_device_name(0)

    # -------------------------------------------------------------------------
    # LINE 4: gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    # -------------------------------------------------------------------------
    # This line has three parts, read from left to right:
    #
    # Part 1: torch.cuda.get_device_properties(0)
    #         -> get all the details of GPU number 0.
    #
    # Part 2: .total_memory
    #         -> from those details, pick out the total memory.
    #         -> it is measured in BYTES, so it is a huge number,
    #            around 15,800,000,000 for a T4.
    #
    # Part 3: / 1024**3
    #         -> convert bytes into GB so the number is readable.
    #         -> 1024**3 means 1024 x 1024 x 1024 = 1,073,741,824
    #         -> bytes / 1024 = KB
    #            KB    / 1024 = MB
    #            MB    / 1024 = GB
    #
    # The final result (about 14.7) is stored in "gpu_memory_gb".
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

    # -------------------------------------------------------------------------
    # LINE 5: print(f"GPU found: {gpu_name} with {gpu_memory_gb:.1f} GB of memory")
    # -------------------------------------------------------------------------
    # - print(...) shows text on the screen.
    # - The letter f before the quotes makes this an "f-string":
    #   anything inside { } is replaced by the value of that variable.
    # - {gpu_name}          -> replaced by "Tesla T4"
    # - {gpu_memory_gb:.1f} -> replaced by the memory number, shown with
    #                          1 digit after the decimal point
    #                          (14.748... becomes 14.7)
    # - Example of what you will see:
    #       GPU found: Tesla T4 with 14.7 GB of memory
    print(f"GPU found: {gpu_name} with {gpu_memory_gb:.1f} GB of memory")


# -----------------------------------------------------------------------------
# LINE 6: else:
# -----------------------------------------------------------------------------
# - This is the other branch of the "if".
# - The indented line below it runs ONLY when torch.cuda.is_available()
#   gave back False, meaning there is no GPU.
else:

    # -------------------------------------------------------------------------
    # LINE 7: print("No GPU found. ...")
    # -------------------------------------------------------------------------
    # - Shows a message telling the person exactly how to switch the GPU on
    #   in Colab.
    # - Nothing crashes here. It is only a warning, so the person can fix
    #   the setting before running the rest of the notebook.
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

## 2.2 Install the libraries

### What we are doing here

Colab already has PyTorch. We add five more libraries from Hugging Face, the company that hosts most open models and datasets. Each one does one job in this notebook:

| Library | Its job in this notebook | First used in |
|---|---|---|
| `transformers` | Downloads the model and tokenizer, and runs the training loop | Part 3 |
| `datasets` | Downloads the training examples and processes them row by row | Part 5 |
| `peft` | Adds LoRA, the small trainable part. "PEFT" = Parameter-Efficient Fine-Tuning | Part 6 |
| `bitsandbytes` | Loads the model in compressed 4-bit form | Part 3.3 |
| `accelerate` | Places the model on the GPU for us | Part 3.3 |

The install cell takes a minute or two. Red warnings about other packages (for example `cudf` or `pyarrow`) are normal on Colab and do not affect this notebook.

The cell after it **imports** the specific tools we will use, and prints the version numbers. If something breaks later, those version numbers are the first thing to check.

In [ ]:
# =============================================================================
# CELL: INSTALL THE LIBRARIES
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Downloads and installs five Python libraries from the internet into this
# Colab session:
#   transformers -> loads models and tokenizers, runs training
#   peft         -> adds LoRA to a model
#   bitsandbytes -> compresses the model to 4 bits
#   datasets     -> downloads and processes training data
#   accelerate   -> places the model on the GPU
#
# Installed libraries stay only for this Colab session. If the session
# restarts or disconnects, run this cell again.
#
# WHAT THE OUTPUT LOOKS LIKE
# --------------------------
# You will see progress bars, and possibly a red "ERROR: pip's dependency
# resolver..." message about other packages such as cudf or pyarrow.
# That message is about packages this notebook does not use. It is safe
# to ignore.
# =============================================================================


# -----------------------------------------------------------------------------
# THE ONLY LINE: !pip install -q -U transformers peft bitsandbytes datasets accelerate
# -----------------------------------------------------------------------------
# Read it piece by piece:
#
#   !        -> Normally a Colab cell runs Python. The "!" at the start says:
#               "run this line as a terminal command instead".
#
#   pip      -> Python's installer. It downloads libraries from the
#               Python Package Index (pypi.org) and installs them.
#
#   install  -> the pip action: install the libraries listed after it.
#
#   -q       -> "quiet": print less text while installing.
#
#   -U       -> "upgrade": if Colab already has an older version of a
#               library, replace it with the newest version.
#
#   transformers peft bitsandbytes datasets accelerate
#            -> the five libraries to install, separated by spaces.
!pip install -q -U transformers peft bitsandbytes datasets accelerate

In [ ]:
# =============================================================================
# CELL: IMPORT THE TOOLS WE NEED
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Installing a library (previous cell) puts it on the computer.
# IMPORTING it (this cell) makes it usable inside this notebook.
#
# There are two ways to import in Python:
#   import X                -> bring in the whole library, use it as X.something
#   from X import Y         -> bring in only Y from library X, use it as Y
#
# At the end, the cell prints the version of each main library.
# If something fails later, these versions help you search for the error.
#
# BEFORE THIS CELL
# ----------------
# "torch" was already imported in the GPU check cell. This cell uses it
# in the first print line, so that cell must have been run.
# =============================================================================


# -----------------------------------------------------------------------------
# Python's built-in tools (these come with Python; nothing to install)
# -----------------------------------------------------------------------------
# math -> maths functions. We use math.log in Part 7 to show how the loss
#         is calculated.
import math

# os   -> tools for files and folders on the computer. We use it in Part 9
#         to list the saved adapter files and measure their size.
import os


# -----------------------------------------------------------------------------
# Whole libraries (used here to print their version numbers)
# -----------------------------------------------------------------------------
# datasets -> the library that downloads and processes training data.
import datasets

# matplotlib.pyplot -> draws charts. "as plt" gives it the short name "plt".
#                      We use it in Part 7 to draw the loss chart.
import matplotlib.pyplot as plt

# peft -> the LoRA library.
import peft

# transformers -> the model and training library.
import transformers


# -----------------------------------------------------------------------------
# Specific tools from the "datasets" library
# -----------------------------------------------------------------------------
# load_dataset -> one function that downloads a dataset from huggingface.co
#                 by its name. Used in Part 5.1.
from datasets import load_dataset


# -----------------------------------------------------------------------------
# Specific tools from the "peft" library (all used for LoRA)
# -----------------------------------------------------------------------------
# LoraConfig                      -> holds the LoRA settings (rank, which tables, ...). Part 6.
# PeftModel                       -> loads a saved adapter back onto a model. Part 9.2.
# get_peft_model                  -> attaches the LoRA tables to a model. Part 6.
# prepare_model_for_kbit_training -> gets a 4-bit model ready for training. Part 6.
#                                    ("kbit" = "a few bits", like our 4 bits.)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training


# -----------------------------------------------------------------------------
# Specific tools from the "transformers" library
# -----------------------------------------------------------------------------
# The brackets ( ) let one import continue over several lines.
from transformers import (
    # AutoModelForCausalLM -> downloads and loads a text-generating model by
    #                         its name. "CausalLM" = predicts the next token.
    #                         Part 3.3.
    AutoModelForCausalLM,

    # AutoTokenizer -> downloads and loads the tokenizer that belongs to a
    #                  model. Part 3.1.
    AutoTokenizer,

    # BitsAndBytesConfig -> holds the 4-bit compression settings. Part 3.3.
    BitsAndBytesConfig,

    # DataCollatorForLanguageModeling -> during training, groups single
    #                  examples into batches, pads them to equal length and
    #                  prepares the correct answers. Part 7.2.
    DataCollatorForLanguageModeling,

    # Trainer -> the ready-made training loop: it feeds batches to the model,
    #            measures the loss and adjusts the trainable numbers. Part 7.
    Trainer,

    # TrainingArguments -> holds the training settings (batch size, learning
    #                      rate, ...). Part 7.2.
    TrainingArguments,
)


# -----------------------------------------------------------------------------
# Print the version of each main library
# -----------------------------------------------------------------------------
# - Every library has a __version__ value, for example "5.18.0".
# - The extra spaces after each name only line the versions up neatly.
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("datasets    ", datasets.__version__)

## 2.3 All the settings in one place

### What we are doing here

Every number you might want to change later is in the next cell, so you never have to hunt through the code. In Colab, the sliders and boxes appear on the right side of the cell. Change a value, then re-run the notebook **from this cell onwards**.

For your first run, leave everything as it is.

| Setting | Default | What it controls | Used in |
|---|---|---|---|
| `NUM_TRAIN_ROWS` | 2000 | How many examples the model learns from | Part 5.3 |
| `NUM_EVAL_ROWS` | 100 | How many examples are kept aside for checking | Part 5.3 |
| `MAX_TOKENS` | 512 | Longest example allowed, in tokens | Part 5.3 |
| `LORA_R` | 16 | The LoRA rank: how many trainable numbers LoRA adds | Part 6 |
| `LEARNING_RATE` | 0.0002 | How big each adjustment is during training | Part 7 |
| `NUM_EPOCHS` | 1 | How many times the model goes through all training examples | Part 7 |

In [ ]:
# =============================================================================
# CELL: ALL THE SETTINGS IN ONE PLACE
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# This cell does not train or download anything.
# It only creates variables (named boxes that hold a value) for every
# setting the rest of the notebook uses.
#
# Keeping all settings in one cell means:
#   - you can see every important choice in one place
#   - to try a different value, you change it here and re-run the notebook
#     from this cell onwards, instead of hunting through the code
#
# WHAT "#@param" MEANS
# --------------------
# Some lines end with "#@param {...}".
# To Python this is only a comment, so it does nothing.
# But Google Colab reads it and draws a slider or input box on the right
# side of the cell, so you can change the value without editing the code.
#   type:"slider"  -> draw a slider
#   min / max      -> the smallest and largest value the slider allows
#   step           -> how much the value jumps each time you move the slider
#   type:"number"  -> draw a box where you can type a number
# The "#@param" part must stay on the same line as its variable,
# otherwise Colab will not draw the slider.
#
# NAMING STYLE
# ------------
# The names are written in CAPITAL LETTERS. This is a Python habit meaning
# "this value is a setting that should not change while the notebook runs".
# =============================================================================


# -----------------------------------------------------------------------------
# MODEL_ID: which model we fine-tune
# -----------------------------------------------------------------------------
# - The name of the model exactly as it appears on huggingface.co.
#   Hugging Face is a website where companies and people share models
#   and datasets for free.
# - "TinyLlama"                -> the account (team) that published it
# - "TinyLlama-1.1B-Chat-v1.0" -> the model itself:
#       1.1B  = 1.1 billion parameters (numbers inside the model)
#       Chat  = this version was already trained to hold conversations
#       v1.0  = version 1.0
# - Later, the notebook uses this name to download the model automatically.
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# -----------------------------------------------------------------------------
# DATASET_ID: which examples we train on
# -----------------------------------------------------------------------------
# - The name of the dataset on huggingface.co.
# - "iamtarun" is the account that published it.
# - It has 18,612 rows. Each row is one task written in English
#   (for example "Write a function to reverse a string")
#   together with the Python code that solves it.
# - These are the examples the model learns from.
DATASET_ID = "iamtarun/python_code_instructions_18k_alpaca"


# -----------------------------------------------------------------------------
# NUM_TRAIN_ROWS: how many examples the model learns from
# -----------------------------------------------------------------------------
# - We use 2,000 of the 18,612 rows, so training finishes in minutes
#   on a free GPU instead of hours.
# - More rows usually means a better result, but a longer training time.
# - Slider: from 500 to 5,000, moving in jumps of 500.
NUM_TRAIN_ROWS = 2000  #@param {type:"slider", min:500, max:5000, step:500}

# -----------------------------------------------------------------------------
# NUM_EVAL_ROWS: how many examples we keep aside for checking
# -----------------------------------------------------------------------------
# - 100 rows that the model NEVER learns from.
# - We use them only to test the model: "How well does it do on examples
#   it has never seen?"
# - If we tested on the training rows, the model could look good simply
#   because it had memorised those exact rows.
# - Slider: from 50 to 300, moving in jumps of 50.
NUM_EVAL_ROWS = 100  #@param {type:"slider", min:50, max:300, step:50}

# -----------------------------------------------------------------------------
# MAX_TOKENS: the longest example we allow
# -----------------------------------------------------------------------------
# - A "token" is a small piece of text (a word, part of a word, or a symbol).
#   The model reads and writes text as tokens.
# - Any example longer than 512 tokens is thrown away before training.
# - Why: long examples use much more GPU memory and slow training down.
# - Slider: from 256 to 1,024, moving in jumps of 128.
MAX_TOKENS = 512  #@param {type:"slider", min:256, max:1024, step:128}

# -----------------------------------------------------------------------------
# LORA_R: the "rank" for LoRA
# -----------------------------------------------------------------------------
# - The "r" from the LoRA explanation: how many patterns the change table
#   is built from.
# - r = 16 means each 2,048 x 2,048 table (4,194,304 numbers) gets a LoRA
#   pair with 65,536 trainable numbers.
# - Bigger r  -> more freedom to change the model, but more numbers to train
#   Smaller r -> fewer numbers to train, but less freedom
# - Slider: from 4 to 64, moving in jumps of 4.
LORA_R = 16  #@param {type:"slider", min:4, max:64, step:4}

# -----------------------------------------------------------------------------
# LEARNING_RATE: how big each adjustment is during training
# -----------------------------------------------------------------------------
# - During training, after every batch of examples, the LoRA numbers are
#   nudged a little to make the predictions less wrong.
#   The learning rate decides how big that nudge is.
# - 2e-4 is scientific notation for 2 x 10^-4 = 0.0002.
# - Too big   -> the numbers jump around and training becomes unstable
#   Too small -> the numbers barely move and the model learns very slowly
# - 0.0002 is a common starting value for LoRA training.
# - Shown as a box where you can type a number.
LEARNING_RATE = 2e-4  #@param {type:"number"}

# -----------------------------------------------------------------------------
# NUM_EPOCHS: how many times the model goes through all training examples
# -----------------------------------------------------------------------------
# - 1 epoch = the model has seen every one of the 2,000 training rows once.
# - 2 epochs = it has seen each row twice, and so on.
# - More epochs -> more learning, but also more risk of memorising the rows
#   instead of learning the general habit.
# - Slider: from 1 to 3, moving in jumps of 1.
NUM_EPOCHS = 1  #@param {type:"slider", min:1, max:3, step:1}


# -----------------------------------------------------------------------------
# ADAPTER_DIR: where we save the result
# -----------------------------------------------------------------------------
# - The name of a folder that will be created in Colab after training.
# - It stores only the LoRA numbers (the "adapter"), about 50 MB,
#   not a full copy of the model (2.2 GB).
ADAPTER_DIR = "tinyllama-python-lora"

# -----------------------------------------------------------------------------
# SYSTEM_PROMPT: the standing instruction given to the model every time
# -----------------------------------------------------------------------------
# - Chat models accept a "system" message: an instruction that sits above
#   the user's question and applies to the whole conversation.
# - We use this exact same sentence in two places:
#     1. inside every training example
#     2. every time we ask the model a question
# - It must be identical in both places. If the model sees a different
#   instruction when we ask questions than it saw during training,
#   its answers get worse.
SYSTEM_PROMPT = "You are a Python coding assistant. Reply with Python code only."


# -----------------------------------------------------------------------------
# Show a confirmation message
# -----------------------------------------------------------------------------
# - print(...) shows text on the screen.
# - The f before the quotes makes this an f-string: anything inside { }
#   is replaced by that variable's value.
# - With the default values, you will see:
#     Training on 2000 examples, checking on 100 examples the model never trains on.
# - This lets you confirm the slider values before moving on.
print(f"Training on {NUM_TRAIN_ROWS} examples, checking on {NUM_EVAL_ROWS} examples the model never trains on.")

---
# Part 3 — Meet TinyLlama

## What we are doing in this part

We load the two things that make up "TinyLlama" and prepare them for training:

1. **3.1 The tokenizer.** The tool that turns text into ID numbers and back. We load it and watch it split one line of Python into tokens.
2. **3.2 The padding fix.** One setting on the tokenizer that, if left alone, would stop the model from ever learning to finish its answers.
3. **3.3 The model, in 4-bit form.** We download TinyLlama (2.2 GB) and compress it to 4 bits while loading, so it uses about 0.7 GB of GPU memory.

**What goes in:** the model name `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.
**What comes out:** a `tokenizer` and a `model`, both ready on the GPU.

## 3.1 The tokenizer: text in, numbers out

A model cannot read letters. It only works with numbers. The **tokenizer** converts text into a list of numbers and back again.

- It cuts text into small pieces called **tokens**. A token can be a whole word, part of a word, or a symbol.
- Each token has a fixed ID number. TinyLlama knows 32,000 different tokens.

Here is the real result for one line of Python, from our test run:

| Text piece | `<s>` | `▁def` | `▁add` | `(` | `a` | `,` | `▁b` | `):` | `▁return` | `▁a` | `▁+` | `▁b` |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **ID** | 1 | 822 | 788 | 29898 | 29874 | 29892 | 289 | 1125 | 736 | 263 | 718 | 289 |

Notice that `▁b` appears twice and gets the same ID, 289, both times. The `▁` symbol means "there was a space before this piece".

There are also a few **special tokens** that are not real words:

| Token | ID | Meaning |
|---|---|---|
| `<s>` | 1 | "the text starts here" |
| `</s>` | 2 | "I am finished" — the **stop token** |
| `<unk>` | 0 | "unknown piece of text" |

The stop token matters a lot today. A model that never produces `</s>` keeps writing until it hits the length limit. Teaching the model **when to stop** is a big part of what fine-tuning will do here.

**Predict first:** how many tokens will `def add(a, b): return a + b` become? Count the words and symbols, then run the cell. (Our answer: 12, including `<s>`.)

In [ ]:
# =============================================================================
# CELL: LOAD THE TOKENIZER AND SEE HOW TEXT BECOMES NUMBERS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# A model cannot read letters. It only works with numbers.
# The TOKENIZER is the tool that converts text into numbers (and back).
#
# It works in two steps:
#   1. It cuts the text into small pieces called TOKENS.
#      A token can be a whole word, part of a word, or a symbol like "(".
#   2. It replaces each token with its ID number.
#      TinyLlama knows exactly 32,000 different tokens, and each one has
#      a fixed ID between 0 and 31,999.
#
# This cell:
#   - downloads TinyLlama's tokenizer
#   - takes one short line of Python code
#   - shows the line as token IDs (what the model really receives)
#   - shows the line as token pieces (so we can read what each ID stands for)
#   - counts how many tokens the line uses
#
# WHY EVERY MODEL HAS ITS OWN TOKENIZER
# -------------------------------------
# Each model was trained with one specific tokenizer.
# The ID 1234 means one particular piece of text for TinyLlama, and
# something completely different for another model.
# So we must always use the tokenizer that belongs to our model.
#
# ABOUT THE WARNING YOU MAY SEE
# -----------------------------
# "You are sending unauthenticated requests to the HF Hub" only means we
# are downloading without logging in to Hugging Face. Downloads still work.
#
# BEFORE THIS CELL
# ----------------
# "AutoTokenizer" was imported in the earlier import cell, and MODEL_ID was
# set in the settings cell. If either cell was skipped, this cell fails.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: load the tokenizer
# -----------------------------------------------------------------------------
# - AutoTokenizer is a helper from the "transformers" library.
#   "Auto" means: you give it a model name, and it works out by itself
#   which kind of tokenizer that model needs.
# - .from_pretrained(MODEL_ID) downloads TinyLlama's tokenizer files from
#   huggingface.co and loads them.
#   "pretrained" means it is the ready-made tokenizer the TinyLlama team
#   created, not a new one.
# - The loaded tokenizer is stored in the variable "tokenizer".
#   We use this same tokenizer for the rest of the notebook.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


# -----------------------------------------------------------------------------
# LINE 2: choose a sample text
# -----------------------------------------------------------------------------
# - A short line of Python code, stored in the variable "example_text".
# - It is only for this demonstration. You can change it to any text
#   and re-run the cell to see how it gets split.
example_text = "def add(a, b): return a + b"


# -----------------------------------------------------------------------------
# LINE 3: convert the text into token IDs
# -----------------------------------------------------------------------------
# Read it in two parts:
#
# Part 1: tokenizer(example_text)
#   - Runs the tokenizer on our text.
#   - It gives back a dictionary (a set of named results). Two of them are:
#       "input_ids"      -> the list of token ID numbers
#       "attention_mask" -> a list of 1s, one per token, meaning
#                           "this is a real token". It is not needed here.
#
# Part 2: ["input_ids"]
#   - Picks out only the list of ID numbers from that dictionary.
#
# - The result is stored in "example_ids".
# - The tokenizer also adds one special token at the very start: <s>,
#   which has ID 1 and means "the text starts here". So the first number
#   in the list is always 1.
# - In our test run the result was:
#     [1, 822, 788, 29898, 29874, 29892, 289, 1125, 736, 263, 718, 289]
example_ids = tokenizer(example_text)["input_ids"]


# -----------------------------------------------------------------------------
# LINE 4: convert the IDs back into readable pieces
# -----------------------------------------------------------------------------
# - convert_ids_to_tokens(...) looks up each ID number and gives back the
#   piece of text it stands for.
# - This lets us see where the tokenizer cut the text.
# - The result is a list of text pieces, stored in "example_pieces".
example_pieces = tokenizer.convert_ids_to_tokens(example_ids)


# -----------------------------------------------------------------------------
# LINE 5: show the original text
# -----------------------------------------------------------------------------
# - print(...) shows on screen whatever is inside the brackets.
# - Here: the label "Text  :" followed by our sample line of code.
print("Text  :", example_text)

# -----------------------------------------------------------------------------
# LINE 6: show the token IDs
# -----------------------------------------------------------------------------
# - Shows the list of ID numbers.
# - This list of numbers is all the model ever receives. It never sees
#   the letters themselves.
print("IDs   :", example_ids)

# -----------------------------------------------------------------------------
# LINE 7: show the token pieces
# -----------------------------------------------------------------------------
# - Shows the same tokens as text pieces, in the same order as the IDs.
# - Things you will notice in the output:
#     "<s>" -> the "text starts here" token added automatically (ID 1)
#     "▁"   -> this special symbol means "there was a space before this
#              piece". For example "▁return" means " return".
#     Some words stay whole; others are split into smaller parts;
#     symbols like "(" "," ")" ":" "+" usually get their own tokens.
print("Pieces:", example_pieces)

# -----------------------------------------------------------------------------
# LINE 8: count the tokens
# -----------------------------------------------------------------------------
# - len(...) counts how many items are in a list.
# - So this shows how many tokens our line of code was cut into,
#   including the <s> token at the start. In our test run: 12.
# - Why this matters:
#     - MAX_TOKENS in the settings cell (512) is counted in tokens, not
#       in letters or words.
#     - More tokens = more GPU memory and more time to process.
print("Count :", len(example_ids), "tokens")

**Try it:** change `example_text` above to your own name, or to a longer line of code, and run the cell again. Notice that the first ID is always `1`. That is `<s>`, which the tokenizer adds by itself at the start.

<details>
<summary><b>Check yourself:</b> why does the model need a tokenizer at all? (click to open)</summary>

Because the model can only do maths on numbers. The tokenizer turns text into a list of ID numbers on the way in, and turns ID numbers back into text on the way out.
</details>

## 3.2 One small fix: the padding token

### What we are doing here

We change one tokenizer setting. Without this one line, the trained model would never learn to stop writing.

### Why it matters, step by step

1. During training we feed several examples to the GPU at once. They have different lengths.
2. So the shorter ones are filled up with a **padding token** until all are equally long.
3. The training code is told to **ignore** padding, because it is not real text.
4. TinyLlama's tokenizer uses the stop token `</s>` as its padding token.
5. So the training code ignores **every** `</s>`, including the real one at the end of each answer.
6. Result: the model is never trained to produce `</s>`, so it never learns to stop.

**The fix:** use `<unk>` (ID 0) as the padding token instead. `<unk>` almost never appears in real text, so it is safe to ignore.

| | Padding token | Real stop token `</s>` at the end of each answer |
|---|---|---|
| Before the fix | `</s>` | Ignored, so the model never learns to stop |
| After the fix | `<unk>` | Learned, so the model learns to stop |

In Part 5.4 we check that every training example really ends with `</s>`.

In [ ]:
# =============================================================================
# CELL: FIX THE PADDING TOKEN
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# It changes one tokenizer setting, the "padding token", so that the model
# can learn WHEN TO STOP writing. Without this fix, the fine-tuned model
# would keep writing until it hits the length limit.
#
# STEP 1: WHAT IS PADDING?
# ------------------------
# During training we give the GPU several examples at the same time
# (a "batch"). The GPU needs every example in a batch to be the SAME length.
# But our examples have different lengths. For example, as token IDs:
#
#     Example A: [1, 822, 788, 29898, 2]          -> 5 tokens
#     Example B: [1, 822, 2]                      -> 3 tokens
#
# So the shorter example is filled up with a filler token until it is as
# long as the longest one. That filler is the PADDING TOKEN:
#
#     Example A: [1, 822, 788, 29898, 2]
#     Example B: [1, 822, 2,   PAD,   PAD]        -> now also 5 tokens
#
# (These examples are made up, only to show the idea.)
#
# STEP 2: PADDING IS IGNORED DURING TRAINING
# ------------------------------------------
# Padding is not real text, so the model must not learn from it.
# The training code therefore marks every padding token as "ignore".
# Whatever the model predicts at those positions does not count.
#
# STEP 3: THE PROBLEM WITH TINYLLAMA
# ----------------------------------
# TinyLlama has a special STOP TOKEN, written </s> (ID 2).
# It means "I am finished". Every training example ends with </s>, so the
# model can learn to produce </s> when its answer is complete.
#
# But TinyLlama's tokenizer uses that SAME </s> as its padding token.
# The training code cannot tell the difference between:
#     - </s> used as padding
#     - </s> used as the real "I am finished" at the end of an answer
# So it ignores BOTH.
#
# Result: the model never learns to produce </s>, so it never learns to
# stop. Its answers run on until the length limit cuts them off.
#
# STEP 4: THE FIX
# ---------------
# Use a different token for padding: <unk> (ID 0).
# <unk> means "unknown piece of text". It almost never appears in real
# text, so it is safe to use as filler.
# Now:
#     - padding (<unk>)      -> ignored, as it should be
#     - real stop token </s> -> NOT ignored, so the model learns to stop
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: show the padding token BEFORE the fix
# -----------------------------------------------------------------------------
# - tokenizer.pad_token gives back the token currently used for padding.
# - It shows </s>, which is the same as the stop token. This is the problem
#   described in Step 3.
print("Padding token before the fix:", tokenizer.pad_token)


# -----------------------------------------------------------------------------
# LINE 2: change the padding token to <unk>
# -----------------------------------------------------------------------------
# - tokenizer.unk_token is TinyLlama's "unknown" token, <unk>.
# - The "=" sign puts that value into tokenizer.pad_token.
# - From now on, short examples are filled with <unk> instead of </s>.
tokenizer.pad_token = tokenizer.unk_token


# -----------------------------------------------------------------------------
# LINE 3: put padding at the END of short examples
# -----------------------------------------------------------------------------
# - Padding can be added either at the start (left) or at the end (right)
#   of a short example:
#
#     "right" -> [1, 822, 2, PAD, PAD]     real tokens first, filler after
#     "left"  -> [PAD, PAD, 1, 822, 2]     filler first, real tokens after
#
# - For training, "right" is the usual choice: the real text starts at
#   position 1 in every example, and the filler comes after it.
tokenizer.padding_side = "right"


# -----------------------------------------------------------------------------
# LINE 4: show the padding token AFTER the fix
# -----------------------------------------------------------------------------
# - Prints the padding token again.
# - It now shows <unk>, which confirms the change worked.
print("Padding token after the fix :", tokenizer.pad_token)

## 3.3 Quantization: making the model fit

### What we are doing here

We download TinyLlama and load it onto the GPU in **4-bit** form. Then we count its parameters and measure the memory it uses.

### The problem

Each of the 1.1 billion parameters is normally stored as a **32-bit** number:

- 1,100,000,000 parameters × 4 bytes each = **about 4.4 GB** just to hold the model.
- Training needs several times more than that on top, for the intermediate calculations.

### The solution: store each number in 4 bits

**Quantization** stores each parameter with fewer bits. We use **4 bits**, which allows only 16 different values.

| | How it is stored | Possible values |
|---|---|---|
| 32-bit | `0.037218...` | about 4 billion different values |
| 4-bit | the nearest of 16 allowed values, for example `0.04` | 16 different values |

### A worked example: compressing 5 real-looking weights

Weights are handled in groups. Here is a group of 5 (the real groups have 64):

`0.037,  −0.012,  0.081,  −0.064,  0.005`

**Step 1. Find the scale:** the largest value ignoring the sign is **0.081**. It is stored once for the whole group.

**Step 2. Divide every weight by 0.081**, so all values are between −1 and +1:
`0.457,  −0.148,  1.000,  −0.790,  0.062`

**Step 3. Replace each with the nearest of 16 levels and store only its 4-bit code.** (For simplicity, these levels are evenly spaced. NF4 places more of them near zero.)

| Original | ÷ 0.081 | Nearest level | Stored code | When used: level × 0.081 |
|---|---|---|---|---|
| 0.037 | 0.457 | 0.467 | 1011 | 0.0378 |
| −0.012 | −0.148 | −0.200 | 0110 | −0.0162 |
| 0.081 | 1.000 | 1.000 | 1111 | 0.0810 |
| −0.064 | −0.790 | −0.733 | 0010 | −0.0594 |
| 0.005 | 0.062 | 0.067 | 1000 | 0.0054 |

The last column is close to the first column, but not exact. That small loss of exactness is the price of storing 8 times less.

### The four settings in the next cell

| Setting | Meaning |
|---|---|
| `load_in_4bit=True` | store each parameter in 4 bits |
| `bnb_4bit_quant_type="nf4"` | choose the 16 allowed values so that more of them sit near zero, where most weights are |
| `bnb_4bit_use_double_quant=True` | also compress the scale numbers (one per group of 64), saving a little more memory |
| `bnb_4bit_compute_dtype=torch.float16` | when calculating, temporarily expand to 16-bit numbers, which the T4 handles quickly |

This is the **Q** in **QLoRA**.

**Predict first:** without compression the model would need about 4.1 GB. How much do you think the 4-bit model will use? (Our test run: **0.70 GB**.)

In [ ]:
# =============================================================================
# CELL: LOAD TINYLLAMA IN 4-BIT (QUANTIZATION)
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Writes down the compression settings (how to store the model in 4 bits).
# 2. Downloads TinyLlama from huggingface.co and loads it onto the GPU,
#    compressing it to 4 bits while it loads.
# 3. Counts the model's parameters and measures how much GPU memory it uses,
#    so we can see how much memory the compression saved.
#
# WHY WE COMPRESS (QUANTIZE)
# --------------------------
# TinyLlama has about 1.1 billion numbers (parameters).
#   - Stored normally, each number uses 32 bits = 4 bytes.
#   - 1.1 billion x 4 bytes = about 4.4 GB, just to hold the model.
#   - Training needs a lot of extra memory on top of that.
# Stored in 4 bits, each number uses 1/8 of the space, so the model takes
# well under 1 GB and there is plenty of room left for training.
#
# HOW 4-BIT STORAGE WORKS (SHORT VERSION)
# ---------------------------------------
# 4 bits allow only 16 patterns (0000 to 1111), so each weight is replaced
# by the closest of 16 allowed levels. Weights are handled in groups of 64,
# and each group stores one "scale" number to turn the levels back into
# real sizes.
#
# WHAT YOU SHOULD SEE
# -------------------
# Download progress bars (about 2.2 GB), then three lines like:
#     Parameters in the model : 1,100,048,384
#     Memory if stored 32-bit : 4.10 GB
#     Memory used right now   : 0.70 GB
#
# BEFORE THIS CELL
# ----------------
# BitsAndBytesConfig and AutoModelForCausalLM were imported in the import
# cell, torch in the GPU check cell, and MODEL_ID was set in the settings cell.
# =============================================================================


# -----------------------------------------------------------------------------
# PART 1: THE COMPRESSION SETTINGS
# -----------------------------------------------------------------------------
# - BitsAndBytesConfig is a settings object from the "transformers" library.
#   It only WRITES DOWN how we want the model compressed. Nothing is
#   compressed yet; that happens in Part 2.
# - "bitsandbytes" is the name of the library that does the actual
#   compression on the GPU.
# - The settings are stored in the variable "bnb_config"
#   (bnb = short for bitsandbytes).
bnb_config = BitsAndBytesConfig(

    # load_in_4bit=True
    # - Store every parameter in 4 bits instead of 32 bits.
    # - This is the main switch that turns compression on.
    load_in_4bit=True,

    # bnb_4bit_quant_type="nf4"
    # - Chooses WHICH 16 levels are allowed.
    # - "nf4" (NormalFloat 4) places more of the 16 levels near zero and
    #   fewer near the edges.
    # - Reason: most weights in a model are very close to zero, so putting
    #   more levels there makes those weights more accurate after rounding.
    # - The other option, "fp4", spaces the levels differently and is
    #   usually a little less accurate for model weights.
    bnb_4bit_quant_type="nf4",

    # bnb_4bit_use_double_quant=True
    # - Every group of 64 weights keeps one "scale" number. With about
    #   1 billion compressed weights, that is about 15 million scale
    #   numbers, and they take memory too.
    # - "Double quant" compresses those scale numbers as well, which saves
    #   a little more memory.
    bnb_4bit_use_double_quant=True,

    # bnb_4bit_compute_dtype=torch.float16
    # - Weights are STORED in 4 bits, but the GPU cannot multiply 4-bit
    #   numbers directly.
    # - So whenever a calculation happens, the GPU temporarily expands the
    #   needed weights to 16-bit numbers, multiplies, and throws the
    #   expanded copy away. Storage stays 4-bit.
    # - "float16" = a 16-bit number with a decimal point. The T4 GPU does
    #   16-bit maths quickly.
    bnb_4bit_compute_dtype=torch.float16,
)


# -----------------------------------------------------------------------------
# PART 2: DOWNLOAD AND LOAD THE MODEL
# -----------------------------------------------------------------------------
# - AutoModelForCausalLM is a helper from the "transformers" library.
#     "Auto"      -> give it a model name and it works out the model type.
#     "CausalLM"  -> a model that predicts the NEXT token from the tokens
#                    before it. TinyLlama is this type of model.
# - .from_pretrained(...) downloads the model files (about 2.2 GB) from
#   huggingface.co the first time, then loads them.
#   "pretrained" = the finished model the TinyLlama team already trained.
# - The loaded model is stored in the variable "model".
model = AutoModelForCausalLM.from_pretrained(

    # MODEL_ID
    # - Which model to download: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    #   (set in the settings cell).
    MODEL_ID,

    # quantization_config=bnb_config
    # - Apply the 4-bit compression settings from Part 1 WHILE loading.
    # - The model never sits in GPU memory at full size; it is compressed
    #   as it loads.
    quantization_config=bnb_config,

    # device_map="auto"
    # - Decides where the model is placed: GPU or CPU.
    # - "auto" = put it on the GPU if there is room (there is, on a T4).
    device_map="auto",
)


# -----------------------------------------------------------------------------
# PART 3: COUNT THE PARAMETERS
# -----------------------------------------------------------------------------
# - model.parameters() goes through every block of numbers in the model,
#   one block at a time (for example, one 2,048 x 2,048 table).
# - p.numel() gives "number of elements": how many numbers are stored in
#   that block.
#
# WHY THE SPECIAL CASE FOR 4-BIT BLOCKS
# - In a 4-bit block, two 4-bit numbers are packed into each byte.
#   So numel() counts the BYTES, which is half the real number of
#   parameters.
# - These 4-bit blocks have the type "Params4bit". For them we multiply
#   the count by 2 to get the real number.
# - Without this fix the cell prints 615,606,272 instead of 1,100,048,384.
# - Blocks that were not compressed (for example the word table at the
#   start) are counted normally.

total_params = 0                                  # start the count at zero
for p in model.parameters():                      # go through every block of numbers
    if p.__class__.__name__ == "Params4bit":      # is this a 4-bit compressed block?
        total_params += p.numel() * 2             # yes: stored count is half, so double it
    else:
        total_params += p.numel()                 # no: stored count is the real count


# -----------------------------------------------------------------------------
# PART 4: MEASURE THE MEMORY USED
# -----------------------------------------------------------------------------
# - model.get_memory_footprint() gives the memory the loaded model takes
#   right now, in bytes.
# - / 1024**3 converts bytes to GB (1024 x 1024 x 1024 = 1,073,741,824).
# - The result is stored in "memory_gb".
memory_gb = model.get_memory_footprint() / 1024**3


# -----------------------------------------------------------------------------
# PART 5: SHOW THE RESULTS
# -----------------------------------------------------------------------------
# Line 1: number of parameters
# - {total_params:,} shows the number with commas, e.g. 1,100,048,384.
print(f"Parameters in the model : {total_params:,}")

# Line 2: memory the model WOULD need without compression
# - total_params * 4   -> 4 bytes per parameter (32-bit storage)
# - / 1024**3          -> convert to GB
# - :.2f               -> show 2 digits after the decimal point
# - Expect about 4.10 GB. (1.1 billion x 4 = 4.4 billion bytes, which is
#   4.10 GB when you divide by 1024 three times instead of 1000.)
print(f"Memory if stored 32-bit : {total_params * 4 / 1024**3:.2f} GB")

# Line 3: memory the model ACTUALLY uses now, in 4-bit
# - Our test run: 0.70 GB. Where it comes from:
#     - about 969 million compressed weights x half a byte = about 0.45 GB
#     - the word table at the start and the word scores at the end
#       (about 131 million numbers) are kept in 16-bit  = about 0.24 GB
#     - the scale numbers                                = a little more
print(f"Memory used right now   : {memory_gb:.2f} GB")

<details>
<summary><b>Check yourself:</b> the model now uses 0.70 GB instead of about 4.1 GB. What did we give up? (click to open)</summary>

A little precision. Each weight is now the nearest of 16 levels instead of its exact value. For most tasks the answers barely change, which you will see in Part 4: the 4-bit TinyLlama still writes sensible Python.
</details>

---
# Part 4 — Before: how does the untouched model answer?

## What we are doing in this part

Before changing anything, we record how TinyLlama answers today. These "before" answers are what we compare against in Part 8.

1. **4.1** See the exact text layout TinyLlama expects for a conversation.
2. **4.2** Write one helper function, `ask`, that sends a question to the model and returns its answer.
3. **4.3** Ask 5 fixed Python questions and save the answers.

**What goes in:** 5 questions.
**What comes out:** 5 answers, stored in a list called `answers_before`.

## 4.1 The chat format

TinyLlama was trained on conversations written in one fixed layout. If we use a different layout, its answers get worse. The layout has three parts:

```
<|system|>
You are a Python coding assistant. Reply with Python code only.</s>
<|user|>
Write a Python function that reverses a string.</s>
<|assistant|>
```

- `<|system|>` — a standing instruction for the whole conversation.
- `<|user|>` — the question.
- `<|assistant|>` — left open. The model continues writing from here. That continuation is its answer.
- `</s>` after each message — "this message is finished".

We do not type this layout by hand. The tokenizer has a function, `apply_chat_template`, that builds it from a simple list of messages.

In [ ]:
# =============================================================================
# CELL: SEE THE CHAT LAYOUT THE MODEL EXPECTS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Writes one short conversation as a plain Python list:
#    a system instruction + one user question.
# 2. Asks the tokenizer to turn that list into TinyLlama's exact chat
#    layout (the <|system|>, <|user|>, <|assistant|> text shown above).
# 3. Prints the result, so we can see exactly what the model will receive.
#
# Nothing is sent to the model yet. This cell only builds and shows text.
#
# WHAT YOU SHOULD SEE
# -------------------
#     <|system|>
#     You are a Python coding assistant. Reply with Python code only.</s>
#     <|user|>
#     Write a Python function that reverses a string.</s>
#     <|assistant|>
#
# BEFORE THIS CELL
# ----------------
# SYSTEM_PROMPT was set in the settings cell, and "tokenizer" was loaded
# in Part 3.1.
# =============================================================================


# -----------------------------------------------------------------------------
# PART 1: the conversation as a Python list
# -----------------------------------------------------------------------------
# - [ ... ] is a Python list: an ordered group of items.
# - Each item is a dictionary { ... }: a set of named values.
#   Every message has two named values:
#     "role"    -> who is speaking: "system", "user" or "assistant"
#     "content" -> the text of the message
# - The list is stored in the variable "demo_messages".
demo_messages = [

    # Message 1: the system instruction (the standing rule for the chat).
    # SYSTEM_PROMPT = "You are a Python coding assistant. Reply with Python code only."
    {"role": "system", "content": SYSTEM_PROMPT},

    # Message 2: the user's question.
    {"role": "user", "content": "Write a Python function that reverses a string."},
]


# -----------------------------------------------------------------------------
# PART 2: turn the list into TinyLlama's chat layout
# -----------------------------------------------------------------------------
# - apply_chat_template(...) is a tokenizer function. TinyLlama's tokenizer
#   carries a "template" that knows exactly how to write each role.
# - The result is stored in "demo_prompt".
demo_prompt = tokenizer.apply_chat_template(

    # demo_messages
    # - The conversation to convert (from Part 1 above).
    demo_messages,

    # tokenize=False
    # - Give us the result as readable TEXT.
    # - With tokenize=True it would give back token ID numbers instead.
    tokenize=False,

    # add_generation_prompt=True
    # - Add "<|assistant|>" at the very end.
    # - This tells the model: "the next text is yours to write".
    # - Without it, the model would not know it is its turn to answer.
    add_generation_prompt=True,
)


# -----------------------------------------------------------------------------
# PART 3: show the result
# -----------------------------------------------------------------------------
# - Prints the exact text the model would receive.
# - Notice the </s> at the end of each message: that is the stop token
#   from Part 3.1, used here to mark the end of each message.
print(demo_prompt)

## 4.2 A helper function to ask the model a question

### What we are doing here

We will ask questions many times in this notebook (Parts 4, 8, 9 and 10), so we write the steps once as a function called `ask`. A function is a named block of code you can run again and again with different inputs.

What `ask` does, in order:

1. **Build** the prompt in the chat layout from 4.1.
2. **Convert** it to token IDs and move them to the GPU.
3. **Generate:** let the model predict new tokens, one at a time, until it produces the stop token `</s>` or reaches 200 new tokens.
4. **Decode:** convert only the new tokens back to text and return them.

We use `do_sample=False`. That means the model always picks its single most likely next token, so the same question gives the same answer every time. This keeps the before-and-after comparison fair.

Running this cell only **defines** the function. Nothing is printed. The function is used in the next cell.

In [ ]:
# =============================================================================
# CELL: DEFINE THE "ask" FUNCTION
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# It creates a reusable function called "ask". Running this cell only
# DEFINES the function; it does not ask anything yet, so nothing is printed.
#
# HOW TO USE IT LATER
# -------------------
#     answer = ask(model, "Write a Python function that reverses a string.")
#
# INSIDE THE FUNCTION, 4 STEPS HAPPEN
# -----------------------------------
#   Step 1: build the prompt in TinyLlama's chat layout
#   Step 2: convert the prompt into token IDs and move them to the GPU
#   Step 3: let the model write new tokens, one at a time
#   Step 4: convert only the NEW tokens back into text, and return it
#
# HOW GENERATION WORKS (STEP 3)
# -----------------------------
# The model does not write the whole answer at once. It repeats a loop:
#   - look at all tokens so far
#   - predict the single next token
#   - add it to the end
#   - repeat
# The loop ends when the model produces the stop token </s>, or after
# 200 new tokens, whichever comes first.
#
# ABOUT A WARNING YOU MAY SEE LATER
# ---------------------------------
# "Both max_new_tokens (=200) and max_length (=2048) seem to have been set"
# appears when the function runs. The model has a default length limit of
# 2,048 tokens, and we also set 200. The message only says our 200 wins.
# It is safe to ignore.
# =============================================================================


# -----------------------------------------------------------------------------
# THE FUNCTION HEADER
# -----------------------------------------------------------------------------
# - "def" starts a function definition. The function is named "ask".
# - It takes three inputs (called parameters):
#     model          -> which model to ask (we will pass the 4-bit TinyLlama,
#                       and later the fine-tuned one)
#     question       -> the question text
#     max_new_tokens -> the longest answer allowed, in tokens.
#                       "=200" makes 200 the default if you do not give one.
def ask(model, question, max_new_tokens=200):

    # The text in triple quotes is a "docstring": a one-line description
    # of what the function does. Python stores it; it does not run.
    """Ask the model one question and return its answer as text."""

    # -------------------------------------------------------------------------
    # STEP 1a: the conversation as a Python list
    # -------------------------------------------------------------------------
    # - Same format as in 4.1: a system instruction and the user's question.
    # - The same SYSTEM_PROMPT is used in training (Part 5), so the model
    #   sees the same instruction at question time as during training.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # -------------------------------------------------------------------------
    # STEP 1b: build the chat layout as text
    # -------------------------------------------------------------------------
    # - Same call as in 4.1:
    #     tokenize=False             -> give back readable text
    #     add_generation_prompt=True -> end with <|assistant|>, so the model
    #                                   knows it should answer next
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # -------------------------------------------------------------------------
    # STEP 2: text -> token IDs, moved to the GPU
    # -------------------------------------------------------------------------
    # - tokenizer(prompt, ...) turns the text into token IDs.
    # - return_tensors="pt" -> give the result as a PyTorch "tensor"
    #   (PyTorch's format for a block of numbers), which the model needs.
    #   "pt" stands for PyTorch.
    # - .to(model.device) -> move those numbers to the same place as the
    #   model (the GPU). The model and its input must be in the same place.
    # - "inputs" now holds input_ids (the token IDs) and attention_mask.
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # -------------------------------------------------------------------------
    # STEP 3: let the model write the answer
    # -------------------------------------------------------------------------
    # - "with torch.no_grad():" means: for the indented lines below, do NOT
    #   keep the extra records that training needs (records of how each
    #   number affected the result). We are only asking, not training, so
    #   skipping them saves memory and time.
    with torch.no_grad():

        # model.generate(...) runs the predict-one-token loop described above.
        # The result, "output_ids", holds the prompt tokens FOLLOWED BY the
        # new answer tokens.
        output_ids = model.generate(

            # **inputs
            # - The two stars unpack the "inputs" dictionary, so this is the
            #   same as writing:
            #       input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]
            **inputs,

            # max_new_tokens=max_new_tokens
            # - Stop after this many new tokens (200 by default), even if
            #   the model has not produced </s>.
            max_new_tokens=max_new_tokens,

            # do_sample=False
            # - At every step, always pick the single most likely next token.
            # - So the same question always gives the same answer, which
            #   keeps the before/after comparison fair.
            # - With do_sample=True the model would pick randomly among
            #   likely tokens, and answers would vary between runs.
            do_sample=False,

            # pad_token_id=tokenizer.pad_token_id
            # - Tells generate which ID is the padding token (<unk>, ID 0,
            #   after our fix in Part 3.2). Without it, generate prints a
            #   warning.
            pad_token_id=tokenizer.pad_token_id,
        )

    # -------------------------------------------------------------------------
    # STEP 4a: find where the answer starts
    # -------------------------------------------------------------------------
    # - inputs["input_ids"] has the shape [1, number_of_prompt_tokens]:
    #   1 question, and N tokens in it.
    # - .shape[1] picks the second number: how many tokens the prompt had.
    prompt_length = inputs["input_ids"].shape[1]

    # -------------------------------------------------------------------------
    # STEP 4b: keep only the new tokens
    # -------------------------------------------------------------------------
    # - output_ids[0]                -> the first (and only) result
    # - [prompt_length:]             -> skip the prompt tokens, keep the rest
    #   Example: if the prompt was 40 tokens and the result is 90 tokens,
    #   this keeps tokens 40 to 89, which is the model's 50-token answer.
    new_ids = output_ids[0][prompt_length:]

    # -------------------------------------------------------------------------
    # STEP 4c: token IDs -> text, and return it
    # -------------------------------------------------------------------------
    # - tokenizer.decode(...) turns token IDs back into text.
    # - skip_special_tokens=True -> leave out <s>, </s> and <unk> from the text.
    # - .strip() removes spaces and empty lines at the start and end.
    # - "return" sends the finished text back to whoever called ask().
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

## 4.3 The five test questions

### What we are doing here

We ask the untouched model 5 simple Python questions and save its answers. These five questions stay the same for the whole notebook: we ask them again in Part 8 after training.

Read the answers and note three things:

- Does the answer contain only code, or also a lot of explanation?
- Does the answer end cleanly, or is it cut off in the middle (meaning it hit the 200-token limit)?
- Is the code correct?

**Predict first:** the system prompt says "Reply with Python code only". Will the untouched model obey it?

<details>
<summary>What happened in our test run (click to open)</summary>

It did not obey. Every answer started with "Here's a Python function that...", wrapped the code in a markdown block, added "Here's an example usage:", and several were cut off mid-sentence at the 200-token limit. Some code was also wrong: the factorial function fails for every input. It keeps calling itself with a smaller number until it reaches 0, and then raises its own error, because it has no case that returns 1.
</details>

In [ ]:
# =============================================================================
# CELL: ASK THE UNTOUCHED MODEL 5 QUESTIONS ("BEFORE" ANSWERS)
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Writes down 5 fixed Python questions.
# 2. Asks the untouched TinyLlama each question, using the "ask" function.
# 3. Saves each answer in a list called "answers_before".
# 4. Prints each question and its answer.
#
# WHY WE SAVE THE ANSWERS
# -----------------------
# In Part 8 we ask the SAME 5 questions to the fine-tuned model and show
# both answers together. Saving them now means we do not need to keep the
# untouched model around.
#
# HOW LONG IT TAKES
# -----------------
# About 1-2 minutes: the model writes up to 200 tokens for each question.
# You may see the harmless "max_new_tokens ... max_length" warning
# explained in the previous cell.
# =============================================================================


# -----------------------------------------------------------------------------
# PART 1: the 5 test questions
# -----------------------------------------------------------------------------
# - A Python list of 5 text items, stored in "TEST_QUESTIONS".
# - Capital letters: this list is fixed and must not change, so the
#   before/after comparison is fair.
# - All 5 are short, single-function tasks that a small model can manage.
TEST_QUESTIONS = [
    "Write a Python function that checks whether a number is prime.",
    "Write a Python function that reverses a string.",
    "Write a Python function that returns the factorial of a number.",
    "Write a Python function that counts how many times each word appears in a sentence.",
    "Write a Python function that returns the largest number in a list without using max().",
]


# -----------------------------------------------------------------------------
# PART 2: an empty list to collect the answers
# -----------------------------------------------------------------------------
# - [] creates an empty list. Each answer will be added to it, in order.
# - After this cell, answers_before[0] is the answer to question 1,
#   answers_before[1] the answer to question 2, and so on.
answers_before = []


# -----------------------------------------------------------------------------
# PART 3: ask every question, save and print the answer
# -----------------------------------------------------------------------------
# - "for ... in ...:" repeats the indented lines once for each question.
# - enumerate(TEST_QUESTIONS, start=1) gives back pairs of
#   (counting number, question), starting the count at 1:
#       1, "Write a Python function that checks whether a number is prime."
#       2, "Write a Python function that reverses a string."
#       ...
#   "number" holds the count and "question" holds the text.
for number, question in enumerate(TEST_QUESTIONS, start=1):

    # Ask the untouched model this question. The answer text is stored
    # in "answer".
    answer = ask(model, question)

    # .append(...) adds the answer to the end of the answers_before list.
    answers_before.append(answer)

    # Show the question number and text. The f-string fills in {number}
    # and {question}.
    print(f"QUESTION {number}: {question}")

    # "-" * 80 repeats the "-" character 80 times: a thin dividing line.
    print("-" * 80)

    # Show the model's answer.
    print(answer)

    # "=" * 80 makes a thick dividing line. "\n" adds an empty line after
    # it, so the questions are visually separated.
    print("=" * 80, "\n")

---
# Part 5 — The training data

## What we are doing in this part

A fine-tuned model can only be as good as its examples. In this part we take a public dataset and turn it into exactly the training material the model needs. Four steps:

1. **5.1 Download and look.** Download 18,612 rows of (instruction, Python code) and read a few, so we know what the model will learn from.
2. **5.2 Rewrite each row as a chat.** Turn every row into the same `<|system|> <|user|> <|assistant|>` layout from Part 4, this time with the correct answer filled in and `</s>` at the end.
3. **5.3 Convert to tokens and split.** Turn each text into token IDs, drop rows longer than 512 tokens, then take 2,000 rows for training and the next 100 for checking.
4. **5.4 Check one example.** Confirm that a training example really ends with the stop token, so the model will learn to stop.

**What goes in:** the dataset name `iamtarun/python_code_instructions_18k_alpaca`.
**What comes out:** `train_dataset` (2,000 rows) and `eval_dataset` (100 rows), both as token IDs.

## What one row of the dataset looks like

| Column | Content | Example |
|---|---|---|
| `instruction` | the task, in English | Create a function to calculate the sum of a sequence of integers. |
| `input` | extra details for the task (often empty) | `[1, 2, 3, 4, 5]` |
| `output` | the Python code that solves it | `def sum_sequence(sequence): ...` |
| `prompt` | the three columns above glued together in a different layout | we do not use it |

## 5.1 Download it and look at a few rows

In [ ]:
# =============================================================================
# CELL: DOWNLOAD THE DATASET AND LOOK AT 3 ROWS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Downloads the training dataset from huggingface.co (about 11 MB).
# 2. Prints a short summary: the column names and the number of rows.
# 3. Prints 3 rows in full, so we can see what the model will learn from.
#
# WHAT YOU SHOULD SEE
# -------------------
#     Dataset({
#         features: ['instruction', 'input', 'output', 'prompt'],
#         num_rows: 18612
#     })
# followed by 3 rows, each with INSTRUCTION, INPUT and OUTPUT.
#
# BEFORE THIS CELL
# ----------------
# load_dataset was imported in the import cell, and DATASET_ID was set in
# the settings cell.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: download the dataset
# -----------------------------------------------------------------------------
# - load_dataset(...) downloads a dataset from huggingface.co by its name.
# - DATASET_ID = "iamtarun/python_code_instructions_18k_alpaca"
# - split="train" -> datasets can come in parts called "splits"
#   (for example "train" and "test"). This dataset has only one split,
#   called "train", which holds all 18,612 rows.
# - The result is stored in "raw_dataset". "raw" because we have not
#   changed anything in it yet.
raw_dataset = load_dataset(DATASET_ID, split="train")


# -----------------------------------------------------------------------------
# LINE 2: print a summary of the dataset
# -----------------------------------------------------------------------------
# - Printing a dataset shows its column names ("features") and its number
#   of rows ("num_rows").
print(raw_dataset)

# -----------------------------------------------------------------------------
# LINE 3: print an empty line
# -----------------------------------------------------------------------------
# - print() with nothing inside prints a blank line, to separate the
#   summary from the rows below.
print()


# -----------------------------------------------------------------------------
# PART 2: print 3 chosen rows
# -----------------------------------------------------------------------------
# - The loop runs once for each row number in the list [0, 2, 9].
#   Row numbers start at 0, so 0 is the first row.
# - These three were picked because they show different kinds of tasks:
#     row 0 -> a function, with an input list
#     row 2 -> a one-line list comprehension, with no input
#     row 9 -> a short script, with an input list
for row_number in [0, 2, 9]:

    # raw_dataset[row_number] fetches one row as a Python dictionary:
    #   {"instruction": "...", "input": "...", "output": "...", "prompt": "..."}
    row = raw_dataset[row_number]

    # Print the task in English.
    print("INSTRUCTION:", row["instruction"])

    # Print the extra details. Often empty, so this line may show nothing
    # after the label.
    print("INPUT      :", row["input"])

    # Print a label on its own line, because the code below it can span
    # several lines.
    print("OUTPUT     :")

    # Print the Python code that solves the task.
    print(row["output"])

    # A thin dividing line between rows.
    print("-" * 80)

**Try it:** change the row numbers in the list above, for example to `[100, 5000, 18000]`, and look at other rows.

**This data is not perfect.** If you browse enough rows you will find:

- rows that are not Python at all (one asks for a Kubernetes file, one for C#),
- code with small mistakes,
- odd values in the `input` column, such as a single `-`.

Real-world training data usually looks like this. With 2,000 rows the model still picks up the main pattern. In a production project you would clean the data first, and that cleaning often takes more time than the training.

## 5.2 Turn each row into one training text

### What we are doing here

The model learns from plain text. So we rewrite every row into the same chat layout from Part 4, this time **with the answer filled in** and the stop token at the end.

**One dataset row:**

| instruction | input | output |
|---|---|---|
| Create a function to calculate the sum of a sequence of integers. | `[1, 2, 3, 4, 5]` | `def sum_sequence(sequence): ...` |

**becomes one training text:**

```
<|system|>
You are a Python coding assistant. Reply with Python code only.</s>
<|user|>
Create a function to calculate the sum of a sequence of integers.

Input:
[1, 2, 3, 4, 5]</s>
<|assistant|>
def sum_sequence(sequence):
    ...
    return sum</s>
```

Compare it with the prompt in Part 4.1. It is the same layout, except that the `<|assistant|>` part is now **filled in** with the correct code. That filled-in part is what the model learns to write.

Three details:

- When the `input` column has real content, we add it under the instruction. When it is empty or says "Not applicable", we leave it out.
- We use the same system prompt as in the `ask` function. Training and asking must use the same layout, otherwise the model sees something at question time that it never saw during training.
- We only process 3,100 rows (2,000 + 100 + 1,000 spare), not all 18,612, because we only need about 2,100. The spare 1,000 cover the rows dropped for being too long in 5.3.

In [ ]:
# =============================================================================
# CELL: REWRITE EACH DATASET ROW INTO THE CHAT LAYOUT
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Defines "build_question": joins a row's instruction and its optional
#    input into one question.
# 2. Defines "to_training_text": turns one row into one complete chat text
#    (system instruction + question + correct answer + stop tokens).
# 3. Shuffles the dataset, keeps 3,100 rows, and runs to_training_text on
#    every one of them.
# 4. Prints one finished training text so we can check it.
#
# WHY SHUFFLE
# -----------
# The rows in the dataset may be stored in some order (for example, similar
# tasks next to each other). Shuffling mixes them, so the 2,000 rows we
# train on are a fair sample of the whole dataset.
#
# WHAT YOU SHOULD SEE
# -------------------
# A progress bar "Map: ... 3100" and then one training text in the
# <|system|> <|user|> <|assistant|> layout, ending with </s>.
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION 1: build_question
# -----------------------------------------------------------------------------
# - Input : one dataset row (a dictionary with "instruction", "input", ...)
# - Output: one question text
def build_question(row):
    """Join the instruction and the optional input into one question."""

    # row["instruction"] -> the task text.
    # .strip()           -> remove spaces and line breaks at the start and end.
    question = row["instruction"].strip()

    # row["input"] -> the extra details (may be empty). Cleaned the same way.
    extra = row["input"].strip()

    # Add the extra details only when they hold real content:
    #   - "extra"                             -> True only if it is not empty
    #   - extra.lower() != "not applicable"   -> and it is not the filler text
    #                                            "Not applicable" (lower()
    #                                            makes the check ignore
    #                                            capital letters)
    if extra and extra.lower() != "not applicable":

        # Put the extra details under the task:
        #   "\n" is a line break, so "\n\n" leaves one empty line.
        # Result looks like:
        #   Create a function to calculate the sum ...
        #
        #   Input:
        #   [1, 2, 3, 4, 5]
        question = question + "\n\nInput:\n" + extra

    # Send the finished question back.
    return question


# -----------------------------------------------------------------------------
# FUNCTION 2: to_training_text
# -----------------------------------------------------------------------------
# - Input : one dataset row
# - Output: a dictionary {"text": the complete chat text}
#   (The dataset library needs a dictionary back, so it can add "text" as
#   a new column.)
def to_training_text(row):
    """Rewrite one dataset row into TinyLlama's chat layout, with the answer included."""

    # One complete conversation, as a Python list of 3 messages.
    messages = [

        # 1. The same standing instruction used in ask(). It must match.
        {"role": "system", "content": SYSTEM_PROMPT},

        # 2. The question, built by the function above.
        {"role": "user", "content": build_question(row)},

        # 3. The correct answer: the Python code from the dataset.
        #    This is the part the model learns to write.
        {"role": "assistant", "content": row["output"].strip()},
    ]

    # Turn the 3 messages into TinyLlama's layout as readable text.
    # - tokenize=False -> give back text, not token IDs.
    # - No add_generation_prompt here: the conversation is already complete,
    #   including the assistant's answer.
    text = tokenizer.apply_chat_template(messages, tokenize=False)

    # The layout ends with "</s>" followed by a line break.
    # .strip() removes that final line break, so </s> (the stop token) is
    # the very last thing in the text. Part 5.4 checks this.
    return {"text": text.strip()}


# -----------------------------------------------------------------------------
# LINE: how many rows to process
# -----------------------------------------------------------------------------
# - NUM_TRAIN_ROWS + NUM_EVAL_ROWS + 1000 = 2,000 + 100 + 1,000 = 3,100
#   (2,100 we need, plus 1,000 spare in case some are too long in 5.3).
# - min(len(raw_dataset), ...) -> never ask for more rows than the dataset
#   has (18,612). This only matters if you set very large slider values.
candidate_count = min(len(raw_dataset), NUM_TRAIN_ROWS + NUM_EVAL_ROWS + 1000)


# -----------------------------------------------------------------------------
# LINE: shuffle, then keep the first 3,100 rows
# -----------------------------------------------------------------------------
# - .shuffle(seed=42) -> mix the rows into a random order.
#   seed=42 fixes the randomness, so everyone who runs this notebook gets
#   the SAME order. (42 is just a commonly used number; any fixed number works.)
# - .select(range(candidate_count)) -> keep rows 0, 1, 2, ... 3,099 of the
#   shuffled order.
#   range(3100) means the numbers 0 to 3,099.
candidates = raw_dataset.shuffle(seed=42).select(range(candidate_count))


# -----------------------------------------------------------------------------
# LINE: run to_training_text on every row
# -----------------------------------------------------------------------------
# - .map(function) runs the function once on every row and adds whatever
#   it returns as new columns. Here it adds a "text" column.
# - The progress bar "Map: 3100" shows it working through the rows.
# - The result is stored in "text_dataset".
text_dataset = candidates.map(to_training_text)


# -----------------------------------------------------------------------------
# LINE: show one finished training text
# -----------------------------------------------------------------------------
# - text_dataset[0] -> the first row after shuffling.
# - ["text"]        -> its new "text" column.
# - Check: does it start with <|system|> and end with </s>?
print(text_dataset[0]["text"])

**Try it:** change `text_dataset[0]` to `text_dataset[7]` and run the cell again to see a different example.

## 5.3 Convert the texts to tokens and drop the very long ones

### What we are doing here

Three things, in this order:

1. **Tokenize:** each training text becomes a list of token IDs (as in Part 3.1).
2. **Filter:** drop every example longer than `MAX_TOKENS` (512).
3. **Split:** the first 2,000 remaining rows become the training set; the next 100 become the checking set.

**Why drop long examples instead of cutting them short?** Cutting removes the end of the text, including the stop token `</s>`. The model would then learn from answers that never finish, which is the opposite of what we want.

In our test run, **2,977 of the 3,100** rows were 512 tokens or shorter. So only 123 were dropped, and we had more than enough for 2,000 + 100.

You may see a warning such as *"Token indices sequence length is longer than the specified maximum sequence length (3909 > 2048)"*. It appears because one very long row is tokenized before being dropped. That row is removed by the filter, so the warning is harmless.

In [ ]:
# =============================================================================
# CELL: TOKENIZE, DROP LONG EXAMPLES, SPLIT INTO TRAINING AND CHECKING SETS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Converts every training text into token IDs.
# 2. Drops every example longer than MAX_TOKENS (512 tokens).
# 3. Takes the first NUM_TRAIN_ROWS (2,000) for training, and the next
#    NUM_EVAL_ROWS (100) for checking.
# 4. Prints how many rows ended up in each part.
#
# WHAT YOU SHOULD SEE
# -------------------
# Two progress bars ("Map" and "Filter"), possibly a harmless warning about
# a sequence longer than 2048, then something like:
#     Rows after dropping long ones: 2977 of 3100
#     Training rows                : 2000
#     Checking rows                : 100
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION: tokenize
# -----------------------------------------------------------------------------
# - Input : one row with a "text" column
# - Output: a dictionary with two lists:
#     "input_ids"      -> the token ID numbers
#     "attention_mask" -> a 1 for every real token (used later to tell real
#                         tokens apart from padding)
def tokenize(row):
    """Convert one training text into token IDs."""
    return tokenizer(row["text"])


# -----------------------------------------------------------------------------
# LINE: tokenize every row
# -----------------------------------------------------------------------------
# - .map(tokenize, ...) runs the tokenize function on every row and adds
#   "input_ids" and "attention_mask" as new columns.
# - remove_columns=text_dataset.column_names
#     -> remove all the OLD columns (instruction, input, output, prompt,
#        text). The training loop only needs the token columns, and extra
#        text columns would get in its way.
# - The result is stored in "tokenized".
tokenized = text_dataset.map(tokenize, remove_columns=text_dataset.column_names)


# -----------------------------------------------------------------------------
# LINE: keep only examples that fit within MAX_TOKENS
# -----------------------------------------------------------------------------
# - .filter(test) keeps only the rows where the test gives True.
# - lambda row: ... is a one-line function written in place:
#     "take a row, and give back True if its number of tokens is 512 or less"
# - len(row["input_ids"]) counts the tokens in that row.
# - The filtered result replaces "tokenized".
tokenized = tokenized.filter(lambda row: len(row["input_ids"]) <= MAX_TOKENS)


# -----------------------------------------------------------------------------
# LINE: the training set
# -----------------------------------------------------------------------------
# - range(NUM_TRAIN_ROWS) = the numbers 0 to 1,999.
# - .select(...) keeps those rows: the first 2,000.
# - The model LEARNS from these rows.
train_dataset = tokenized.select(range(NUM_TRAIN_ROWS))


# -----------------------------------------------------------------------------
# LINE: the checking set
# -----------------------------------------------------------------------------
# - range(NUM_TRAIN_ROWS, NUM_TRAIN_ROWS + NUM_EVAL_ROWS)
#   = range(2000, 2100) = the numbers 2,000 to 2,099.
# - So these 100 rows come right after the training rows and never overlap
#   with them.
# - The model NEVER learns from these. We use them only to measure how well
#   it does on examples it has not seen.
eval_dataset = tokenized.select(range(NUM_TRAIN_ROWS, NUM_TRAIN_ROWS + NUM_EVAL_ROWS))


# -----------------------------------------------------------------------------
# Show the counts
# -----------------------------------------------------------------------------
# Line 1: how many rows survived the length filter, out of how many we had.
print(f"Rows after dropping long ones: {len(tokenized)} of {len(text_dataset)}")

# Line 2: how many rows the model will learn from.
print(f"Training rows                : {len(train_dataset)}")

# Line 3: how many rows are kept aside for checking.
print(f"Checking rows                : {len(eval_dataset)}")

**Why two sets?** The model adjusts its numbers using the training rows. To know whether it learned the general pattern, and did not just memorise those exact rows, we measure it on rows it has never trained on. Those are the checking rows (the usual name is the **evaluation set** or **validation set**).

## 5.4 Check one example

### What we are doing here

A quick safety check before training: the last token of a training example must be the stop token `</s>` (ID 2). If it is not, the model will never learn to stop, and the whole run is wasted. The check uses `assert`, which stops the notebook with an error message if the condition is false.

In [ ]:
# =============================================================================
# CELL: CHECK THAT A TRAINING EXAMPLE ENDS WITH THE STOP TOKEN
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Takes the first training example (as token IDs).
# 2. Prints its length, its first token and its last token.
# 3. Checks that the last token is the stop token </s> (ID 2).
#    If not, the notebook stops here with an error message.
#
# WHAT YOU SHOULD SEE
# -------------------
#     Number of tokens: 110            (the exact number may differ)
#     First token ID  : 1 -> <s>
#     Last token ID   : 2 -> </s>
#     Check passed: the example ends with the stop token.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: get the token IDs of the first training example
# -----------------------------------------------------------------------------
# - train_dataset[0]   -> the first training row
# - ["input_ids"]      -> its list of token IDs
sample_ids = train_dataset[0]["input_ids"]


# -----------------------------------------------------------------------------
# LINE 2: how many tokens the example has
# -----------------------------------------------------------------------------
print("Number of tokens:", len(sample_ids))

# -----------------------------------------------------------------------------
# LINE 3: the first token
# -----------------------------------------------------------------------------
# - sample_ids[0] -> the first ID in the list. Should be 1.
# - convert_ids_to_tokens(...) shows which token that ID stands for.
#   Should be <s> ("the text starts here").
print("First token ID  :", sample_ids[0], "->", tokenizer.convert_ids_to_tokens(sample_ids[0]))

# -----------------------------------------------------------------------------
# LINE 4: the last token
# -----------------------------------------------------------------------------
# - sample_ids[-1] -> the LAST ID in the list. In Python, -1 means
#   "count from the end". Should be 2.
# - Should show </s> ("I am finished").
print("Last token ID   :", sample_ids[-1], "->", tokenizer.convert_ids_to_tokens(sample_ids[-1]))


# -----------------------------------------------------------------------------
# LINE 5: the actual check
# -----------------------------------------------------------------------------
# - assert CONDITION, "message"
#     -> if CONDITION is True, nothing happens and the cell continues.
#     -> if CONDITION is False, Python stops with an error showing "message".
# - tokenizer.eos_token_id is the ID of the stop token </s>, which is 2.
#   ("eos" = "end of sequence")
assert sample_ids[-1] == tokenizer.eos_token_id, "The example does not end with the stop token."

# -----------------------------------------------------------------------------
# LINE 6: confirmation
# -----------------------------------------------------------------------------
# - Only reached if the assert above passed.
print("Check passed: the example ends with the stop token.")

---
# Part 6 — LoRA: train 1% of the numbers instead of 100%

## What we are doing in this part

We prepare the model for training in a way that fits on a free GPU:

1. **Freeze** all 1.1 billion original numbers, so none of them can change.
2. **Attach** small trainable tables (LoRA) next to 154 of the model's big tables: 7 tables in each of the 22 layers.
3. **Print** how many numbers can now be trained. In our test run: **12,615,680**, which is **1.13%** of the model.

**What goes in:** the 4-bit model from Part 3.3.
**What comes out:** the same model with LoRA attached, where only the LoRA numbers can change.

## 6.1 The problem LoRA solves

Fine-tuning means changing the model's numbers. The normal way, called **full fine-tuning**, changes all 1.1 billion of them. That causes two problems:

1. **It does not fit in memory.** While training, the computer keeps about 4 numbers for every number it is changing: the number itself, how much to nudge it, and two tracking values. 1.1 billion × 4 = 4.4 billion numbers, about **17.6 GB**. The T4 has about 15 GB.
2. **The result is huge.** You end up with a whole new copy of the model (over 2 GB) for one small change in behaviour.

LoRA solves both. It trains only about **1%** of the numbers and leaves the rest untouched.

## 6.2 The main idea: new table = old table + change table

Inside the model, the numbers sit in tables (rows and columns). Instead of rewriting a table, we keep the old table and add a second table that says how much to add to each number:

```
new table = old table + change table
```

- **Full fine-tuning** learns every number in the change table one by one.
- **LoRA** builds the change table from **two short lists**, which have far fewer numbers.

## 6.3 How two short lists make a whole table

Take a small table of 4 rows × 4 columns, which holds 16 numbers. Write down two short lists of 4 numbers each, 8 numbers in total:

```
List 1 (going down):    1, 2, 0, -1
List 2 (going across):  0.1, 0, 0.2, -0.1
```

Rule: **take each number in list 1 and multiply it by every number in list 2. That gives one row.**

```
1  × [0.1, 0, 0.2, -0.1] = [ 0.1,  0,  0.2, -0.1]   ← row 1
2  × [0.1, 0, 0.2, -0.1] = [ 0.2,  0,  0.4, -0.2]   ← row 2
0  × [0.1, 0, 0.2, -0.1] = [ 0,    0,  0,    0  ]   ← row 3
-1 × [0.1, 0, 0.2, -0.1] = [-0.1,  0, -0.2,  0.1]   ← row 4
```

**8 numbers produced a full table of 16 numbers.** That table is the change table. Add it to the old table and you get the new table. The old table itself was never edited.

## 6.4 What "low rank" means

Look at the 4 rows above. They are all the **same pattern** `[0.1, 0, 0.2, -0.1]`, multiplied by 1, 2, 0 or -1.

- **Rank** = how many different patterns a table is built from. The table above is built from 1 pattern, so its rank is 1.
- A fully free 4 × 4 table could have 4 completely different rows: rank 4, and it needs all 16 numbers.
- **Low rank** = we allow only a few patterns, so there are fewer numbers to learn.

The researchers who invented LoRA found that the change needed to teach a model one new task can be built from only a few patterns. That is the name: **Lo**w-**R**ank **A**daptation.

## 6.5 The same idea at TinyLlama's real size

One table in TinyLlama, called `q_proj`, is 2,048 rows × 2,048 columns = **4,194,304** numbers.

| | Numbers trained for this one table |
|---|---|
| Full fine-tuning | 4,194,304 |
| LoRA with 1 pattern (rank 1): one list of 2,048 down + one of 2,048 across | 4,096 |
| **LoRA with 16 patterns (rank 16, our setting)** | **16 × 4,096 = 65,536 (1.6%)** |

In the code, the 16 lists going across are stored together as table **A** (16 × 2,048), and the 16 lists going down as table **B** (2,048 × 16).

## 6.6 What happens to one input

For an input `x` (a list of 2,048 numbers for one token):

```
x ──► old table W (frozen) ───────────────────────────────► result 1
x ──► A (16 × 2,048) ──► 16 numbers ──► B (2,048 × 16) ──► result 2 × 2

output = result 1 + 2 × result 2
```

- `W · x` is the original model's result, unchanged.
- `B · A · x` is the correction learned during training.
- The `× 2` is a fixed multiplier, `lora_alpha / r` = 32 / 16 = 2.

At the start, **B is filled with zeros**. So result 2 is zero, and the model behaves **exactly** like the original. You will see this in Part 7.3: the starting loss is the original model's loss. Training then moves A and B away from zero, step by step.

## 6.7 Full fine-tuning vs LoRA

| | Full fine-tuning | LoRA (our run) |
|---|---|---|
| Numbers trained | about 1,100,000,000 | 12,615,680 (1.13%) |
| Fits on a free T4 | no | easily |
| What you save at the end | a full copy of the model (over 2 GB) | only A and B (about 48 MB) |
| Original model | overwritten | untouched, reusable for other tasks |

## 6.8 The settings in the next cell

| Setting | Our value | Meaning |
|---|---|---|
| `r` | 16 | the rank: how many patterns. Larger means more trainable numbers |
| `lora_alpha` | 32 | sets the multiplier `alpha / r` = 2 |
| `lora_dropout` | 0.05 | during training, randomly switch off 5% of the LoRA inputs. This discourages memorising |
| `target_modules` | 7 table names | which tables inside each layer get an A and B pair |
| `task_type` | `CAUSAL_LM` | tells the library this model predicts the next token |

**Predict first:** each of the 22 layers has 7 tables. How many A-and-B pairs will LoRA add? (Answer: 22 × 7 = **154**.)

In [ ]:
# =============================================================================
# CELL: FREEZE THE MODEL AND ATTACH LoRA
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Switches off a feature that is only useful when generating answers.
# 2. Prepares the 4-bit model for training: freezes all its original
#    numbers so none of them can change.
# 3. Writes down the LoRA settings.
# 4. Attaches an A-and-B pair of small trainable tables next to each of the
#    7 chosen table types, in all 22 layers (154 pairs in total).
# 5. Prints how many numbers can now be trained.
#
# WHAT YOU SHOULD SEE
# -------------------
#     trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338
#
# WHERE 12,615,680 COMES FROM (with r = 16)
# -----------------------------------------
# For each table, LoRA adds 16 x (rows + columns) numbers:
#     q_proj    2,048 x 2,048 -> 16 x (2,048 + 2,048) =  65,536
#     k_proj    2,048 x   256 -> 16 x (2,048 +   256) =  36,864
#     v_proj    2,048 x   256 -> 16 x (2,048 +   256) =  36,864
#     o_proj    2,048 x 2,048 -> 16 x (2,048 + 2,048) =  65,536
#     gate_proj 2,048 x 5,632 -> 16 x (2,048 + 5,632) = 122,880
#     up_proj   2,048 x 5,632 -> 16 x (2,048 + 5,632) = 122,880
#     down_proj 5,632 x 2,048 -> 16 x (5,632 + 2,048) = 122,880
#                                         one layer  = 573,440
#     22 layers x 573,440 = 12,615,680
# And "all params" = 1,100,048,384 (original) + 12,615,680 (LoRA)
#                  = 1,112,664,064
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: switch off the generation cache
# -----------------------------------------------------------------------------
# - When a model GENERATES text, it keeps a "cache": it remembers
#   calculations from earlier tokens so it does not redo them for every
#   new token. That makes answering faster.
# - During TRAINING the cache is not used, and leaving it on only causes
#   warnings. So we switch it off here, and switch it back on in Part 8.
model.config.use_cache = False


# -----------------------------------------------------------------------------
# LINE 2: prepare the 4-bit model for training
# -----------------------------------------------------------------------------
# - prepare_model_for_kbit_training(...) is a helper from the "peft" library.
#   ("kbit" = "a few bits", like our 4-bit model.)
# - It does three things:
#     1. FREEZES every original number in the model, so training cannot
#        change it.
#     2. Moves a few small, sensitive parts of the model (for example the
#        normalisation numbers in each layer) to 32-bit, so training stays
#        numerically stable.
#     3. Sets up the model so that LoRA tables attached later CAN be trained.
# - The prepared model replaces "model".
model = prepare_model_for_kbit_training(

    # model
    # - The 4-bit TinyLlama loaded in Part 3.3.
    model,

    # use_gradient_checkpointing=False
    # - "Gradient checkpointing" saves GPU memory during training by
    #   throwing away some intermediate results and recalculating them
    #   when needed. That makes training about 20-30% slower.
    # - TinyLlama is small enough to fit without it, so we switch it off
    #   and train faster.
    use_gradient_checkpointing=False,
)


# -----------------------------------------------------------------------------
# LINE 3: the LoRA settings
# -----------------------------------------------------------------------------
# - LoraConfig(...) only WRITES DOWN the settings. Nothing is attached yet.
# - The settings are stored in "lora_config".
lora_config = LoraConfig(

    # r=LORA_R
    # - The rank: how many patterns each change table is built from.
    #   LORA_R = 16 from the settings cell.
    # - So table A is 16 x (columns) and table B is (rows) x 16.
    r=LORA_R,

    # lora_alpha=LORA_R * 2
    # - 16 x 2 = 32.
    # - The LoRA correction is multiplied by lora_alpha / r = 32 / 16 = 2.
    # - Keeping alpha = 2 x r is a common habit, so the multiplier stays 2
    #   even if you change r.
    lora_alpha=LORA_R * 2,

    # lora_dropout=0.05
    # - During training only: for each batch, randomly switch off 5% of the
    #   numbers going INTO the LoRA tables.
    # - This stops LoRA from relying too heavily on any single number, which
    #   helps against memorising the training rows.
    # - It is switched off automatically when the model answers questions.
    lora_dropout=0.05,

    # target_modules=[...]
    # - The names of the tables, inside each of the 22 layers, that get an
    #   A-and-B pair. These 7 are all the big tables in a TinyLlama layer:
    #     q_proj, k_proj, v_proj, o_proj        -> the 4 tables of the
    #                                              "attention" part, which
    #                                              decides which earlier
    #                                              tokens matter for the
    #                                              next one
    #     gate_proj, up_proj, down_proj         -> the 3 tables of the
    #                                              "feed-forward" part, which
    #                                              processes each token
    #                                              further
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

    # bias="none"
    # - Some tables have a short extra list of numbers called a "bias".
    #   "none" = do not train any of them. Only A and B are trained.
    bias="none",

    # task_type="CAUSAL_LM"
    # - Tells peft what kind of model this is: one that predicts the next
    #   token ("causal language model"). peft uses this to attach LoRA in
    #   the right way.
    task_type="CAUSAL_LM",
)


# -----------------------------------------------------------------------------
# LINE 4: attach LoRA to the model
# -----------------------------------------------------------------------------
# - get_peft_model(model, lora_config) goes through the model, finds every
#   table named in target_modules (7 per layer x 22 layers = 154 tables),
#   and places an A-and-B pair next to each one.
# - Table A starts with small random numbers; table B starts as all zeros,
#   so at first the model behaves exactly like the original.
# - The model with LoRA attached replaces "model".
model = get_peft_model(model, lora_config)


# -----------------------------------------------------------------------------
# LINE 5: show how many numbers can be trained
# -----------------------------------------------------------------------------
# - Prints three numbers:
#     trainable params -> numbers in all A and B tables (these will change)
#     all params       -> every number in the model, frozen + trainable
#     trainable%       -> trainable / all x 100
model.print_trainable_parameters()

Read the line printed above:

- `trainable params` is the count of numbers in all the A and B tables together. In our test run: **12,615,680**.
- `all params` is every number in the model: the 1,100,048,384 frozen ones plus the trainable ones.
- `trainable%` should be close to **1%**. In our test run: 1.1338%.

**Try it later:** change `LORA_R` in Part 2.3 to 4 or to 64, re-run from there, and see how the trainable count changes.

<details>
<summary><b>Check yourself:</b> if you set LORA_R to 32, how many trainable numbers will there be? (click to open)</summary>

Every LoRA table doubles in width, so the count doubles: 2 × 12,615,680 = **25,231,360**.
</details>

---
# Part 7 — Training

## What we are doing in this part

This is where the model actually learns. Five steps:

1. **7.1** Understand what happens in one training step, and what the **loss** number means.
2. **7.2** Write down the training settings and set up the ready-made training loop (`Trainer`).
3. **7.3** Measure the loss on the 100 checking rows **before** training: our starting number.
4. **7.4** Train: go through the 2,000 examples once. In our test run this took **6.6 minutes**.
5. **7.5** Draw the loss as a chart, and measure the loss on the checking rows **after** training.

**What goes in:** the model with LoRA attached, and 2,000 training examples.
**What comes out:** a trained model, and a loss that dropped from **1.514 to 0.585** in our test run.

## 7.1 What happens in one training step

1. Take a small group of examples. This group is called a **batch**.
2. For every position in every example, the model predicts the next token.
3. Compare each prediction with the token that actually comes next in the example. The result is one number, the **loss**. A lower loss means better predictions.
4. Work out, for each number in A and B, whether nudging it up or down would lower the loss.
5. Nudge each of them a tiny amount in that direction. The size of the nudge is the **learning rate**.

Then repeat with the next batch. One full pass through all training examples is called an **epoch**.

### A concrete example of step 2 and 3

Take the training text `... <|assistant|>\ndef reverse(s):\n    return s[::-1]</s>`.

At the position right after `def`, the correct next token is `▁reverse`. The model gives a probability to every one of its 32,000 tokens. Suppose it gives `▁reverse` a probability of 0.10. Then the loss at that position is −log(0.10) = 2.30. If after training it gives `▁reverse` 0.90, the loss there drops to 0.11.

### The loss, with real numbers

For one prediction, the loss is `-log(probability the model gave to the correct token)`. Run the cell below to see what that gives.

In [ ]:
# =============================================================================
# CELL: SEE HOW THE LOSS IS CALCULATED
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# For one prediction, the loss is:
#     loss = -log(probability the model gave to the correct next token)
#
# This cell calculates that loss for 5 example probabilities, so you can
# see the pattern:
#     high probability for the correct token -> loss close to 0   (good)
#     low probability for the correct token  -> large loss         (bad)
#
# WHAT YOU SHOULD SEE
# -------------------
#     ... probability of 0.01  ->  loss = 4.61
#     ... probability of 0.10  ->  loss = 2.30
#     ... probability of 0.50  ->  loss = 0.69
#     ... probability of 0.90  ->  loss = 0.11
#     ... probability of 0.99  ->  loss = 0.01
#
# HOW TO READ OUR TRAINING NUMBERS WITH THIS
# ------------------------------------------
# The loss during training is the AVERAGE over all tokens.
#   - A loss of 1.514 (our "before") is about the loss you get when the
#     correct token gets probability 0.22 (since -log(0.22) = 1.51).
#   - A loss of 0.585 (our "after") is about probability 0.56.
# =============================================================================


# -----------------------------------------------------------------------------
# THE LOOP
# -----------------------------------------------------------------------------
# - The loop runs once for each of the 5 probabilities in the list.
# - Each time, "probability" holds one of them.
for probability in [0.01, 0.10, 0.50, 0.90, 0.99]:

    # math.log(x) is the natural logarithm of x.
    # For any number between 0 and 1 it is negative, so the minus sign
    # in front turns the loss into a positive number.
    loss = -math.log(probability)

    # Show the probability and its loss, each with 2 digits after the
    # decimal point (":.2f").
    print(f"Model gives the correct token a probability of {probability:.2f}  ->  loss = {loss:.2f}")

So a loss near 0 means the model is almost always sure of the correct next token, and a higher loss means it is often wrong or unsure. The loss reported during training is the average over all tokens in the batch.

**Try it:** add `0.25` to the list in the cell above. Its loss should be close to our "before" number of 1.514.

## 7.2 The training settings

### What we are doing here

We write down every training setting, create the **data collator** (which builds batches), and create the **Trainer** (the ready-made training loop that runs all the steps in 7.1 for us). Nothing trains yet.

| Setting | Our value | Meaning |
|---|---|---|
| `num_train_epochs` | 1 | go through the training examples once |
| `per_device_train_batch_size` | 4 | 4 examples go through the GPU at a time |
| `gradient_accumulation_steps` | 4 | collect the result of 4 such groups before making one nudge, so each nudge is based on 4 × 4 = 16 examples |
| `learning_rate` | 0.0002 | the size of each nudge |
| `lr_scheduler_type` | cosine | make the nudges gradually smaller towards the end |
| `warmup_steps` | 10 | for the first 10 nudges, grow the learning rate from 0 up to its full value, to avoid a rough start |
| `logging_steps` | 10 | print the training loss every 10 nudges |
| `eval_strategy`, `eval_steps` | steps, 25 | every 25 nudges, measure the loss on the checking rows |
| `fp16` | True | do the maths in 16-bit numbers, which is faster on a T4 |
| `save_strategy` | no | do not save intermediate copies; we save once at the end |
| `report_to` | none | do not send the numbers to any outside tracking service |

With 2,000 examples and 16 examples per nudge, one epoch is 2,000 ÷ 16 = **125 nudges** (the output calls them *steps*). The checking loss is measured at steps 25, 50, 75, 100 and 125.

**Why not just use a batch of 16?** Each example in a batch takes GPU memory at the same time. 4 at a time fits comfortably. Collecting 4 groups before nudging gives the same result as a batch of 16, without needing the memory for 16 at once.

In [ ]:
# =============================================================================
# CELL: SET UP THE TRAINING (SETTINGS, BATCH BUILDER, TRAINING LOOP)
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. TrainingArguments  -> writes down every training setting.
# 2. DataCollator       -> the tool that builds each batch from single
#                          examples during training.
# 3. Trainer            -> the ready-made training loop, given the model,
#                          the settings, the data and the batch builder.
#
# Nothing trains in this cell, and nothing is printed. Training starts in
# Part 7.4 with trainer.train().
#
# THE NUMBERS THAT FOLLOW FROM THESE SETTINGS
# -------------------------------------------
#   examples per nudge = 4 (batch) x 4 (accumulation)       = 16
#   nudges per epoch   = 2,000 examples / 16                = 125 steps
#   training loss shown at steps 10, 20, 30, ... 120
#   checking loss shown at steps 25, 50, 75, 100, 125
# =============================================================================


# -----------------------------------------------------------------------------
# PART 1: the training settings
# -----------------------------------------------------------------------------
# - TrainingArguments(...) only stores settings. Stored in "training_args".
training_args = TrainingArguments(

    # output_dir="training-output"
    # - A folder the Trainer needs for its own files. Because we set
    #   save_strategy="no" below, it stays almost empty.
    output_dir="training-output",

    # num_train_epochs=NUM_EPOCHS
    # - How many passes over the 2,000 training examples. 1 by default.
    num_train_epochs=NUM_EPOCHS,

    # per_device_train_batch_size=4
    # - How many examples go through the GPU at the same time.
    # - "per_device" = per GPU. We have one GPU.
    per_device_train_batch_size=4,

    # gradient_accumulation_steps=4
    # - Run 4 batches of 4, add up how each would nudge the numbers, and
    #   only then make ONE nudge. So each nudge is based on 16 examples.
    # - "Gradient" = the record of which way, and how strongly, each
    #   number should be nudged to lower the loss.
    gradient_accumulation_steps=4,

    # learning_rate=LEARNING_RATE
    # - The size of each nudge: 0.0002 from the settings cell.
    learning_rate=LEARNING_RATE,

    # lr_scheduler_type="cosine"
    # - The learning rate does not stay fixed. After warm-up it slowly
    #   shrinks, following the shape of a cosine curve, down to almost 0
    #   at the last step. Big nudges early, fine adjustments at the end.
    lr_scheduler_type="cosine",

    # warmup_steps=10
    # - For the first 10 nudges, the learning rate rises from 0 up to
    #   0.0002. At the very start, the nudges can point in poor directions;
    #   starting small avoids an unstable first few steps.
    warmup_steps=10,

    # logging_steps=10
    # - Show the training loss every 10 nudges.
    logging_steps=10,

    # eval_strategy="steps"
    # - Measure the loss on the checking rows at regular step intervals
    #   (the other options are "no" and "epoch").
    eval_strategy="steps",

    # eval_steps=25
    # - That interval: every 25 nudges.
    eval_steps=25,

    # per_device_eval_batch_size=8
    # - When CHECKING, there is no nudging, so less memory is needed per
    #   example. We can process 8 at a time, which is faster.
    per_device_eval_batch_size=8,

    # save_strategy="no"
    # - Do not save copies of the model during training. We save once,
    #   ourselves, in Part 9.
    save_strategy="no",

    # fp16=True
    # - Do the training maths in 16-bit numbers instead of 32-bit.
    #   Faster on a T4 and uses less memory.
    fp16=True,

    # report_to="none"
    # - The Trainer can send its numbers to outside tracking websites
    #   (for example Weights & Biases). "none" keeps everything inside
    #   this notebook.
    report_to="none",

    # seed=42
    # - Fixes the random choices made during training (for example which
    #   numbers dropout switches off), so a re-run gives the same result.
    seed=42,
)


# -----------------------------------------------------------------------------
# PART 2: the batch builder (data collator)
# -----------------------------------------------------------------------------
# For every batch during training, the data collator does three things:
#   1. Takes 4 examples of different lengths.
#   2. Pads the shorter ones with <unk> (our padding token from Part 3.2)
#      so all 4 are the same length.
#   3. Creates the "labels": the correct tokens the model's predictions are
#      compared with. The labels are a copy of the input tokens, with every
#      padding position replaced by -100, which means "ignore this position".
#      The model then compares its prediction at each position with the
#      label at the NEXT position.
data_collator = DataCollatorForLanguageModeling(

    # tokenizer=tokenizer
    # - Needed to know which token is the padding token (<unk>).
    tokenizer=tokenizer,

    # mlm=False
    # - "mlm" = masked language modelling: a different training style where
    #   random words in the middle are hidden and the model guesses them.
    # - False = use "predict the next token" instead, which is how TinyLlama
    #   works.
    mlm=False,
)


# -----------------------------------------------------------------------------
# PART 3: the training loop
# -----------------------------------------------------------------------------
# - Trainer(...) ties everything together. It runs the 5 steps from 7.1
#   over and over. Stored in "trainer".
trainer = Trainer(

    # model=model
    # - The model with LoRA attached (Part 6). Only the A and B tables
    #   will change.
    model=model,

    # args=training_args
    # - The settings from Part 1 above.
    args=training_args,

    # train_dataset=train_dataset
    # - The 2,000 rows to learn from (Part 5.3).
    train_dataset=train_dataset,

    # eval_dataset=eval_dataset
    # - The 100 rows to check on (Part 5.3). Never learned from.
    eval_dataset=eval_dataset,

    # data_collator=data_collator
    # - How to build each batch (Part 2 above).
    data_collator=data_collator,
)

**What the data collator does to each batch, in one example:**

| | Position 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| Example (token IDs) | 1 | 822 | 788 | 2 | `<unk>` (padding) |
| Label | 1 | 822 | 788 | 2 | −100 (ignore) |

The model's prediction at each position is compared with the label at the **next** position. Because padding is `<unk>` and not `</s>`, the real stop token (2) keeps its label and is learned. This is where the padding fix from Part 3.2 pays off.

## 7.3 The starting loss

### What we are doing here

Before any training, we measure the average loss on the 100 checking rows. Because B starts as all zeros, the LoRA correction is zero, so this is the loss of the **original TinyLlama**. It is our "before" number.

**Predict first:** will the starting loss be closer to 0 or closer to 5? (Our test run: **1.514**. The original model already predicts code reasonably well, but not in our exact format.)

In [ ]:
# =============================================================================
# CELL: MEASURE THE LOSS BEFORE TRAINING
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Runs the model over the 100 checking rows, WITHOUT changing anything,
# and reports the average loss. This is the "before" number we compare
# with after training.
#
# WHY THIS IS THE ORIGINAL MODEL'S LOSS
# -------------------------------------
# LoRA's table B starts as all zeros, so the LoRA correction is zero.
# The model with LoRA attached therefore behaves exactly like the original
# TinyLlama at this point.
#
# WHAT YOU SHOULD SEE
# -------------------
# A small progress table, then a line like:
#     Loss on the checking rows BEFORE training: 1.514
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: run the check and read the loss
# -----------------------------------------------------------------------------
# - trainer.evaluate() goes through eval_dataset (100 rows) in batches of 8,
#   calculates the loss for every token, and returns a dictionary such as:
#     {"eval_loss": 1.514, "eval_runtime": ..., ...}
# - ["eval_loss"] picks out the average loss.
# - Stored in "loss_before" so we can compare it later.
loss_before = trainer.evaluate()["eval_loss"]

# -----------------------------------------------------------------------------
# LINE 2: show it
# -----------------------------------------------------------------------------
# - :.3f shows 3 digits after the decimal point.
print(f"Loss on the checking rows BEFORE training: {loss_before:.3f}")

## 7.4 Train

### What we are doing here

We run the training: 125 nudges, each based on 16 examples. Only the LoRA tables A and B change. In our test run this took **6.6 minutes**.

Run the next cell and watch the table that appears:

- **Step** — how many nudges have been made so far (out of 125).
- **Training Loss** — the loss on the batches the model is learning from.
- **Validation Loss** — the loss on the checking rows. This is the more trustworthy of the two, because the model never learns from those rows.

Both should fall quickly in the first steps and then level off. While it runs, do not close the browser tab.

In [ ]:
# =============================================================================
# CELL: TRAIN THE MODEL
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Starts the training loop and waits until it finishes. For every one of
# the 125 steps, the Trainer:
#   1. takes 4 batches of 4 examples (16 examples)
#   2. lets the model predict every next token in them
#   3. calculates the loss
#   4. works out which way each LoRA number should be nudged
#   5. nudges every LoRA number a tiny amount
# Every 10 steps it shows the training loss; every 25 steps it measures the
# loss on the 100 checking rows.
#
# Then it prints how long training took.
#
# HOW LONG IT TAKES
# -----------------
# Our test run on a Colab T4: 6.6 minutes for 2,000 examples, 1 epoch.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: train
# -----------------------------------------------------------------------------
# - trainer.train() runs the full training described above.
# - While it runs, a live table of Step / Training Loss / Validation Loss
#   appears under the cell.
# - When it finishes, it returns a summary object, stored in "train_result".
#   train_result.metrics holds numbers such as the total time in seconds.
train_result = trainer.train()


# -----------------------------------------------------------------------------
# LINE 2: training time in minutes
# -----------------------------------------------------------------------------
# - train_result.metrics["train_runtime"] -> the training time in SECONDS.
# - / 60 -> convert seconds to minutes.
minutes = train_result.metrics["train_runtime"] / 60

# -----------------------------------------------------------------------------
# LINE 3: show it
# -----------------------------------------------------------------------------
# - :.1f -> 1 digit after the decimal point, e.g. "6.6".
print(f"Training finished in {minutes:.1f} minutes.")

## 7.5 The loss as a chart

### What we are doing here

The Trainer kept a log of every loss it showed. We pull the numbers out of that log, draw them as two lines, and then measure the checking loss one final time to compare with the "before" number from 7.3.

**Predict first:** the loss started at 1.514. Where will it end? (Our test run: **0.585**.)

In [ ]:
# =============================================================================
# CELL: DRAW THE LOSS CHART AND COMPARE BEFORE vs AFTER
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Reads the Trainer's log of every loss it reported during training.
# 2. Separates it into two lists:
#      - training loss  (reported every 10 steps)
#      - checking loss  (reported every 25 steps)
# 3. Draws both as lines on one chart.
# 4. Measures the checking loss once more, and prints it next to the
#    "before" number from Part 7.3.
#
# WHAT YOU SHOULD SEE
# -------------------
# A chart with two lines falling from left to right, then:
#     Loss on the checking rows BEFORE training: 1.514
#     Loss on the checking rows AFTER training : 0.585
# (Your numbers will be close to these.)
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: the Trainer's log
# -----------------------------------------------------------------------------
# - trainer.state.log_history is a list of dictionaries, one per report.
#   Examples of entries:
#     {"loss": 0.82, "step": 10, ...}         <- a training loss report
#     {"eval_loss": 0.71, "step": 25, ...}    <- a checking loss report
history = trainer.state.log_history


# -----------------------------------------------------------------------------
# LINES 2-5: pull out the numbers for the chart
# -----------------------------------------------------------------------------
# Each line is a "list comprehension": a one-line way to build a list.
# Read it as: "for every entry in history, if the entry has this key,
#             take this value".

# The step numbers where a training loss was reported (10, 20, 30, ...).
train_steps = [entry["step"] for entry in history if "loss" in entry]

# The training loss values at those steps.
train_losses = [entry["loss"] for entry in history if "loss" in entry]

# The step numbers where the checking loss was measured (25, 50, 75, 100, 125).
eval_steps = [entry["step"] for entry in history if "eval_loss" in entry]

# The checking loss values at those steps.
eval_losses = [entry["eval_loss"] for entry in history if "eval_loss" in entry]


# -----------------------------------------------------------------------------
# DRAW THE CHART (matplotlib, imported as "plt")
# -----------------------------------------------------------------------------
# Start a new chart, 8 units wide and 4 units high.
plt.figure(figsize=(8, 4))

# Draw the training loss as a line: steps along the bottom, loss going up.
# label=... is the name shown in the legend.
plt.plot(train_steps, train_losses, label="Training loss")

# Draw the checking loss as a second line. marker="o" puts a dot at each
# measured point, because there are only 5 of them.
plt.plot(eval_steps, eval_losses, marker="o", label="Loss on checking rows")

# Label the horizontal axis.
plt.xlabel("Training step")

# Label the vertical axis.
plt.ylabel("Loss (lower is better)")

# Give the chart a title.
plt.title("Loss during fine-tuning")

# Show a small box naming each line.
plt.legend()

# Add light grid lines. alpha=0.3 makes them 30% visible, so they do not
# distract.
plt.grid(alpha=0.3)

# Display the finished chart under the cell.
plt.show()


# -----------------------------------------------------------------------------
# MEASURE THE CHECKING LOSS AFTER TRAINING
# -----------------------------------------------------------------------------
# - Same as Part 7.3, but now A and B have been trained.
# - Stored in "loss_after".
loss_after = trainer.evaluate()["eval_loss"]

# Show the "before" number from Part 7.3.
print(f"Loss on the checking rows BEFORE training: {loss_before:.3f}")

# Show the new number. Lower = the model now predicts our examples better.
print(f"Loss on the checking rows AFTER training : {loss_after:.3f}")

**How to read the chart**

- Both lines go down → the model is learning the pattern.
- Training loss keeps going down while the checking loss goes **up** → the model has started memorising the training rows instead of learning the general pattern. This is called **overfitting**. With one epoch on 2,000 rows you will usually not see it. Try 3 epochs on 500 rows if you want to see it happen.

**What the drop means in our test run:** the loss went from 1.514 to 0.585. Using the table in 7.1, that is roughly the move from giving the correct next token a probability of about 0.22 to about 0.56, on rows the model never trained on.

<details>
<summary><b>Check yourself:</b> why do we trust the checking loss more than the training loss? (click to open)</summary>

Because the model never learned from the checking rows. A falling training loss could just mean the model is memorising its 2,000 examples. A falling checking loss means it does better on new examples too.
</details>

---
# Part 8 — After: the same five questions

## What we are doing in this part

We ask the trained model the **same 5 questions** from Part 4, with the same `ask` function and the same settings. The only thing that changed is the trained A and B tables. Then we show each "before" and "after" answer one below the other, and compare their lengths.

**What goes in:** the 5 questions.
**What comes out:** 5 new answers in `answers_after`, shown next to `answers_before`.

**Predict first:** will the answers still start with "Here's a Python function..."?

In [ ]:
# =============================================================================
# CELL: ASK THE TRAINED MODEL THE SAME 5 QUESTIONS, SHOW BEFORE vs AFTER
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Switches the model from training mode to answering mode.
# 2. Asks the trained model the same 5 questions as in Part 4.3.
# 3. For each question, prints the BEFORE answer (from Part 4.3) and the
#    AFTER answer, one below the other.
#
# HOW LONG IT TAKES
# -----------------
# Usually faster than Part 4.3, because the answers are shorter and the
# model now stops by itself.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: switch to answering mode
# -----------------------------------------------------------------------------
# - model.eval() switches off training-only behaviour, mainly the LoRA
#   dropout (Part 6), so no numbers are randomly switched off while
#   answering.
model.eval()

# -----------------------------------------------------------------------------
# LINE 2: switch the generation cache back on
# -----------------------------------------------------------------------------
# - We switched it off in Part 6 for training. Generating is faster with it.
model.config.use_cache = True


# -----------------------------------------------------------------------------
# LINE 3: an empty list for the new answers
# -----------------------------------------------------------------------------
answers_after = []


# -----------------------------------------------------------------------------
# LOOP 1: ask every question
# -----------------------------------------------------------------------------
# - For each of the 5 questions, ask the trained model and add the answer
#   to answers_after. The answers are kept in the same order as the questions.
for question in TEST_QUESTIONS:
    answers_after.append(ask(model, question))


# -----------------------------------------------------------------------------
# LOOP 2: show BEFORE and AFTER for each question
# -----------------------------------------------------------------------------
# - enumerate(..., start=1) gives (1, question 1), (2, question 2), ...
for number, question in enumerate(TEST_QUESTIONS, start=1):

    # The question.
    print(f"QUESTION {number}: {question}")

    # A heading line: 34 dashes, the word BEFORE, 38 dashes.
    print("-" * 34, "BEFORE", "-" * 38)

    # The answer saved in Part 4.3.
    # Lists count from 0, so question 1 is at position 0: number - 1.
    print(answers_before[number - 1])

    # A heading line for the new answer.
    print("-" * 34, "AFTER", "-" * 39)

    # The answer from the trained model.
    print(answers_after[number - 1])

    # A thick dividing line and an empty line between questions.
    print("=" * 80, "\n")

## 8.1 One simple measurement: answer length

### What we are doing here

A quick way to see the change in habit is to count the characters in each answer. Shorter usually means "code only, then stop".

In our test run:

| Question | Before | After |
|---|---|---|
| 1. Prime check | 535 characters | 210 characters |
| 2. Reverse a string | 693 | 189 |
| 3. Factorial | 420 | 121 |
| 4. Word count | 662 | 294 |
| 5. Largest number | 468 | 199 |

In [ ]:
# =============================================================================
# CELL: COMPARE ANSWER LENGTHS, BEFORE vs AFTER
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# For each of the 5 questions, counts the characters in the "before" answer
# and the "after" answer, and prints both on one line.
#
# WHAT YOU SHOULD SEE
# -------------------
#     Question 1:  535 characters before  ->   210 characters after
#     ...
# (Your numbers may differ a little.)
# =============================================================================


# -----------------------------------------------------------------------------
# THE LOOP
# -----------------------------------------------------------------------------
# - range(len(TEST_QUESTIONS)) = range(5) = the positions 0, 1, 2, 3, 4.
for number in range(len(TEST_QUESTIONS)):

    # len(text) counts the characters in a piece of text.
    before_length = len(answers_before[number])

    # Same for the answer after training.
    after_length = len(answers_after[number])

    # Show both.
    # - {number + 1}      -> positions start at 0, questions start at 1
    # - {before_length:4d} -> show the whole number using 4 character
    #                         spaces, so the columns line up
    print(f"Question {number + 1}: {before_length:4d} characters before  ->  {after_length:4d} characters after")

**What to look for when you compare**

- Is the "after" answer mostly code, with little or no explanation around it?
- Does it end cleanly instead of being cut off at the limit?
- Is the code right? Copy one answer into a new cell and run it.

**What happened in our test run:**

- **The format changed completely.** Every "after" answer was plain code, with no "Here's a Python function..." and no markdown, and every one stopped by itself.
- **Some code got better.** The "before" factorial function failed for every input. The "after" one was correct: `if n == 0: return 1`, otherwise `n * factorial(n-1)`.
- **Some code got worse.** The "after" answer to *reverse a string* copied the string character by character **without reversing it**. It looks right at a glance and is wrong.

**Be honest with your audience about the limits.** This is a 1.1-billion-parameter model trained for a few minutes on 2,000 rows. What you should see is a clear change in *how* it answers. It has not become a reliable coding assistant, and its code still has to be tested.

---
# Part 9 — Save what the model learned

## What we are doing in this part

1. **9.1** Save only the trained A and B tables (the **adapter**) to a folder, and list the files with their sizes.
2. **9.2** Load a fresh, untouched copy of TinyLlama, put the saved adapter on top, and ask one question, to prove the save worked.
3. **9.3** Optionally download the adapter to your laptop or upload it to your Hugging Face account, because Colab deletes all files when the session ends.

**What goes in:** the trained model.
**What comes out:** a folder called `tinyllama-python-lora`, about 52 MB.

## 9.1 Save the adapter

We save only the A and B tables. Together they are called the **adapter**. The original TinyLlama is not saved again, because it did not change and anyone can download it.

**Why about 48 MB?** The adapter holds 12,615,680 numbers. They are stored in 32-bit (4 bytes each), because they were trained at full precision: 12,615,680 × 4 bytes ≈ **48 MB**. The tokenizer files we save next to it add about 3.5 MB. Compare that with **2.2 GB** for the full model.

In [ ]:
# =============================================================================
# CELL: SAVE THE ADAPTER AND LIST THE FILES
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Saves the trained LoRA tables (A and B) and their settings into the
#    folder ADAPTER_DIR ("tinyllama-python-lora").
# 2. Saves the tokenizer into the same folder, including our padding fix.
# 3. Lists every file in the folder with its size, and the folder total.
#
# WHAT YOU SHOULD SEE (our test run)
# ----------------------------------
#     README.md                     0.00 MB
#     adapter_config.json           0.00 MB   <- the LoRA settings (r, alpha, ...)
#     adapter_model.safetensors    48.16 MB   <- the A and B numbers
#     chat_template.jinja           0.00 MB   <- the chat layout from Part 4.1
#     tokenizer.json                3.45 MB   <- the tokenizer
#     tokenizer_config.json         0.00 MB   <- tokenizer settings
#
#     Whole adapter folder: 51.6 MB
#
# The folder is in Colab's file panel (the folder icon on the left).
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: save the LoRA tables
# -----------------------------------------------------------------------------
# - model.save_pretrained(folder) on a model with LoRA attached saves ONLY
#   the LoRA part: the A and B numbers (adapter_model.safetensors) and the
#   LoRA settings (adapter_config.json). The frozen original numbers are
#   not saved.
# - ".safetensors" is a safe, fast file format for storing model numbers.
model.save_pretrained(ADAPTER_DIR)

# -----------------------------------------------------------------------------
# LINE 2: save the tokenizer
# -----------------------------------------------------------------------------
# - Saves the tokenizer next to the adapter, so whoever loads it later uses
#   the same tokenizer, with <unk> as the padding token.
tokenizer.save_pretrained(ADAPTER_DIR)


# -----------------------------------------------------------------------------
# LINE 3: a running total for the folder size
# -----------------------------------------------------------------------------
total_bytes = 0


# -----------------------------------------------------------------------------
# THE LOOP: list every file and its size
# -----------------------------------------------------------------------------
# - os.listdir(folder) gives the names of all files in the folder.
# - sorted(...) puts them in alphabetical order.
for file_name in sorted(os.listdir(ADAPTER_DIR)):

    # os.path.join(folder, name) builds the full path, for example
    # "tinyllama-python-lora/adapter_config.json".
    # os.path.getsize(path) gives that file's size in bytes.
    file_bytes = os.path.getsize(os.path.join(ADAPTER_DIR, file_name))

    # Add this file's size to the running total.
    total_bytes += file_bytes

    # Show the file name and size.
    # - {file_name:35s}               -> the name, padded to 35 characters,
    #                                    so the sizes line up in a column
    # - {file_bytes / 1024**2:8.2f}   -> bytes converted to MB
    #                                    (1024 x 1024), 2 decimal places,
    #                                    8 characters wide
    print(f"{file_name:35s} {file_bytes / 1024**2:8.2f} MB")


# -----------------------------------------------------------------------------
# LINE 4: the folder total
# -----------------------------------------------------------------------------
# - "\n" at the start adds an empty line before the total.
# - Compare this with the 2.2 GB full model.
print(f"\nWhole adapter folder: {total_bytes / 1024**2:.1f} MB")

## 9.2 Load it back, the way someone else would

### What we are doing here

This is what a colleague does to use your fine-tuned model:

1. Load the original TinyLlama (same as Part 3.3). This copy knows nothing about our training.
2. Load your saved adapter on top of it.
3. Ask a question.

If the answer below matches the "after" answer to question 2 from Part 8, the save and load worked. (In our test run it matched exactly, including the same mistake.)

This loads a second copy of TinyLlama, which uses about 0.7 GB more GPU memory. The T4 has plenty of room for that.

In [ ]:
# =============================================================================
# CELL: LOAD THE SAVED ADAPTER ONTO A FRESH COPY OF TINYLLAMA
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Loads a brand-new, untouched copy of TinyLlama in 4-bit, exactly as in
#    Part 3.3.
# 2. Loads the adapter we saved in 9.1 and attaches it to that copy.
# 3. Asks question 2 ("reverse a string") and prints the answer.
#
# HOW TO KNOW IT WORKED
# ---------------------
# The answer printed here should be the same as the AFTER answer to
# question 2 in Part 8. Same answer = the adapter was saved and loaded
# correctly.
# =============================================================================


# -----------------------------------------------------------------------------
# STEP 1: a fresh copy of the original model
# -----------------------------------------------------------------------------
# - Exactly the same call as Part 3.3, stored under a NEW name,
#   "fresh_base", so it does not replace our trained "model".
fresh_base = AutoModelForCausalLM.from_pretrained(

    # The same base model: TinyLlama-1.1B-Chat-v1.0.
    MODEL_ID,

    # The same 4-bit compression settings from Part 3.3.
    quantization_config=bnb_config,

    # On the GPU.
    device_map="auto",
)


# -----------------------------------------------------------------------------
# STEP 2: attach the saved adapter
# -----------------------------------------------------------------------------
# - PeftModel.from_pretrained(base_model, folder)
#     -> reads adapter_config.json to see where the A and B tables go,
#     -> reads adapter_model.safetensors for their numbers,
#     -> attaches them to fresh_base.
# - The result is stored in "reloaded_model".
reloaded_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)

# -----------------------------------------------------------------------------
# STEP 3: answering mode
# -----------------------------------------------------------------------------
# - Switch off training-only behaviour (dropout), as in Part 8.
reloaded_model.eval()


# -----------------------------------------------------------------------------
# STEP 4: ask question 2 and show the answer
# -----------------------------------------------------------------------------
# - TEST_QUESTIONS[1] -> the SECOND question (lists count from 0):
#   "Write a Python function that reverses a string."
print(ask(reloaded_model, TEST_QUESTIONS[1]))

## 9.3 Keep a copy (optional)

### What we are doing here

Colab deletes all files when the session ends, including the adapter folder. Pick one of these two ways to keep it.

**Option A — download a zip to your laptop.** Remove the `#` at the start of each code line in the next cell and run it.

**Option B — upload to your Hugging Face account.** You need a free account and an access token with *write* permission (huggingface.co → Settings → Access Tokens). Remove the `#` signs in the cell after that, and put your own username in the repository name.

For a seminar, do this once the day before. If the live training is slow or Colab disconnects, you can load the saved adapter with the code from 9.2 and continue from Part 8.

In [ ]:
# =============================================================================
# CELL (OPTIONAL): DOWNLOAD THE ADAPTER AS A ZIP FILE
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Packs the adapter folder into one .zip file and downloads it through
# your browser to your laptop.
#
# HOW TO USE IT
# -------------
# Every code line below starts with "# ", so right now Python skips all of
# them and the cell does nothing. To use it, delete the "# " at the start
# of the 4 code lines (not the explanation lines), then run the cell.
#
# TO USE THE ZIP LATER
# --------------------
# Upload the zip to a new Colab session, unzip it, then run Part 9.2.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: import shutil
# -----------------------------------------------------------------------------
# - shutil is a built-in Python library for copying, moving and zipping
#   files and folders.
# import shutil

# -----------------------------------------------------------------------------
# LINE 2: import Colab's file helper
# -----------------------------------------------------------------------------
# - google.colab.files only exists inside Colab. Its download() function
#   sends a file from Colab to your browser.
# from google.colab import files

# -----------------------------------------------------------------------------
# LINE 3: zip the folder
# -----------------------------------------------------------------------------
# - shutil.make_archive(name, "zip", folder)
#     -> creates "tinyllama-python-lora.zip" containing everything in the
#        adapter folder.
# shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)

# -----------------------------------------------------------------------------
# LINE 4: download the zip
# -----------------------------------------------------------------------------
# - ADAPTER_DIR + ".zip" = "tinyllama-python-lora.zip"
# - Your browser shows a normal download.
# files.download(ADAPTER_DIR + ".zip")

In [ ]:
# =============================================================================
# CELL (OPTIONAL): UPLOAD THE ADAPTER TO YOUR HUGGING FACE ACCOUNT
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Logs in to your Hugging Face account and uploads the adapter and the
# tokenizer to a repository (a named storage space) under your username.
# After that, anyone you share it with (or you, in any Colab session) can
# load it with:
#     PeftModel.from_pretrained(fresh_base, "YOUR_USERNAME/tinyllama-python-lora")
#
# BEFORE USING IT
# ---------------
# 1. Create a free account on huggingface.co.
# 2. Go to Settings -> Access Tokens and create a token with WRITE permission.
# 3. Replace YOUR_USERNAME below with your Hugging Face username.
# 4. Delete the "# " at the start of the 4 code lines, then run the cell.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: import the login helper
# -----------------------------------------------------------------------------
# - huggingface_hub is installed together with transformers.
# from huggingface_hub import login

# -----------------------------------------------------------------------------
# LINE 2: log in
# -----------------------------------------------------------------------------
# - Shows a box where you paste your access token.
# login()

# -----------------------------------------------------------------------------
# LINE 3: upload the adapter
# -----------------------------------------------------------------------------
# - Creates the repository "YOUR_USERNAME/tinyllama-python-lora" if it does
#   not exist, and uploads the A and B tables and their settings.
# model.push_to_hub("YOUR_USERNAME/tinyllama-python-lora")

# -----------------------------------------------------------------------------
# LINE 4: upload the tokenizer to the same repository
# -----------------------------------------------------------------------------
# tokenizer.push_to_hub("YOUR_USERNAME/tinyllama-python-lora")

---
# Part 10 — Try your own question

## What we are doing in this part

Type any Python task into the box on the right of the next cell, and run it. The trained model answers live. Ask the audience for suggestions.

Simple, single-function tasks work best. Try a harder one too, so everyone can see where a model this small reaches its limit.

Ideas to try:

- *Write a Python function that checks whether a word is a palindrome.* (our test run: correct)
- *Write a Python function that returns the second largest number in a list.*
- *Write a Python function that removes duplicates from a list but keeps the order.*
- *Write a Python class for a bank account with deposit and withdraw methods.* (harder)

In [ ]:
# =============================================================================
# CELL: ASK YOUR OWN QUESTION
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Sends one question of your choice to the trained model and prints the
# answer.
#
# HOW TO USE IT
# -------------
# - In Colab, a text box appears on the right side of this cell
#   (because of "#@param {type:"string"}").
# - Type your question in the box, then run the cell.
# - Or edit the text between the quotes directly.
#
# After printing, copy the answer into a new cell and run it, to check
# whether the code actually works.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: the question
# -----------------------------------------------------------------------------
# - #@param {type:"string"} tells Colab to show a text box for this value.
my_question = "Write a Python function that checks whether a word is a palindrome."  #@param {type:"string"}

# -----------------------------------------------------------------------------
# LINE 2: ask the trained model and show the answer
# -----------------------------------------------------------------------------
# - ask(model, my_question) uses the function from Part 4.2, with the
#   trained model.
print(ask(model, my_question))

---
# Part 11 — Recap

## What we did, with our test-run numbers

1. Loaded TinyLlama in 4-bit form, so it used **0.70 GB** of GPU memory instead of about 4 GB (**quantization**, the Q in QLoRA).
2. Asked it 5 Python questions and kept the answers. They were chatty, wrapped in markdown, and often cut off.
3. Rewrote 2,000 (instruction, code) pairs into TinyLlama's chat layout, each ending with the stop token.
4. Froze the original model and added small trainable tables to it (**LoRA**): **12,615,680** numbers, 1.13% of the model.
5. Trained for one pass over the data in **6.6 minutes** and watched the checking loss fall from **1.514 to 0.585**.
6. Asked the same 5 questions: every answer became plain code that stopped by itself, between 2 and 4 times shorter.
7. Saved the result as an adapter of about **48 MB** and loaded it back onto a fresh copy of TinyLlama.

## What we did not do

| Not done | Why it matters |
|---|---|
| Full fine-tuning of all 1.1 billion parameters | It needs more memory and time. LoRA gets most of the benefit for a small task like this |
| Cleaning the dataset | Some rows are not Python or contain mistakes. Cleaner data gives a better model |
| Testing whether the generated code runs correctly | We only read the answers and measured the loss. A proper test runs each answer against test cases and reports the percentage that pass. Our "reverse a string" answer shows why: it looked right and was wrong |
| Training only on the answer part | The loss here is measured on the whole text, including the system prompt and the question. Measuring it on the answer tokens only is a common refinement |
| Merging the adapter into the base model | We keep them separate. Merging produces one standalone model, which is simpler to deploy |

## Words you now know

| Word | Meaning |
|---|---|
| Parameter / weight | one of the numbers that make up the model |
| Token | a small piece of text with an ID number |
| Tokenizer | the tool that turns text into token IDs and back |
| Stop token (`</s>`) | the token that means "I am finished" |
| Padding | filler tokens that make examples in a batch the same length |
| Quantization | storing each parameter with fewer bits |
| LoRA | freezing the model and training small added tables |
| Rank (`r`) | how many patterns the LoRA change table is built from |
| QLoRA | quantization + LoRA together |
| Adapter | the saved LoRA tables |
| Batch | the group of examples processed together |
| Epoch | one full pass over the training examples |
| Loss | how wrong the predictions are; lower is better |
| Learning rate | the size of each adjustment |
| Checking (evaluation) set | examples kept aside to test on, never learned from |
| Overfitting | memorising the training rows instead of learning the general pattern |

## Exercises

1. **More data.** Set `NUM_TRAIN_ROWS` to 4000. Does the final loss on the checking rows improve? How much longer does training take?
2. **Smaller LoRA.** Set `LORA_R` to 4. How many trainable parameters are there now? Are the answers noticeably worse?
3. **See overfitting.** Set `NUM_TRAIN_ROWS` to 500 and `NUM_EPOCHS` to 3. Watch the two lines in the loss chart.
4. **Change the habit.** Change `SYSTEM_PROMPT` so the model must add a one-line comment above every function, then think about what the training data would need to look like for that to work.
5. **Your own data.** Write 20 (instruction, code) pairs in the style of your own team's code. What would you need to change in Part 5 to train on them?